# Notebook 1: Huấn luyện Unpaired F-LSeSim (`RGB 3-ch → IR 1-ch`) & Đánh giá kép so sánh với CycleGAN

Notebook độc lập, tự chứa toàn bộ mã nguồn (**thuần PyTorch / Torchvision**, upload thẳng lên **Kaggle / Colab / Server Local** không cần `git clone`) thực hiện trọn vẹn luồng thay thế **CycleGAN** bằng **F-LSeSim** (*Spatially-Correlative Loss*, Zheng et al., CVPR 2021):

### Checklist trước khi bấm `Run All` / `Save & Run All` trên Kaggle:
1. **Bật GPU** (*Accelerator: GPU T4 x2 hoặc P100*) và **bật Internet** (*Settings -> Internet -> On* để tải trọng số VGG16 / InceptionV3 chuẩn của Torchvision và `pycocotools` ở lần đầu).
2. **Gắn (Add Input) 3 nguồn dữ liệu vào Kaggle Notebook** và điền đường dẫn vào **Cell 1 (`CONFIG`)**:
   - Dataset FLIR aligned (`data_root`, chứa `align_train.txt`, `align_validation.txt`, `JPEGImages/`, `Annotations/`).
   - Trọng số CycleGAN hiện tại (`cyclegan_weights`, ví dụ `latest_net_G_A.pth`).
   - Trọng số FCOS Detector tốt nhất của bạn (`best_fcos_checkpoint`, hỗ trợ tự động cả `best_model.pth` từ `example_flir.py` lẫn `best_iter*.pth` từ `training_full_d3t_v2_rtc.py`).
   - *(Mẹo: Nếu bạn chưa nhớ chính xác đường dẫn trên `/kaggle/input`, cứ chạy **Cell 1 + Cell 2** trước — Cell 2 sẽ tự động quét `/kaggle/input` và in sẵn đường dẫn thực tế để bạn copy-paste!)*

### Các cam kết kỹ thuật nghiêm ngặt trong Notebook này:
1. **100% Unpaired / Disjoint trên FLIR (`align_train.txt`):**
   - Nửa đầu (`N // 2` $\approx 2.064$ stem) **chỉ đọc ảnh RGB 3 kênh** làm Miền A (`trainA`).
   - Nửa sau ($\approx 2.065$ stem) **chỉ đọc ảnh IR 1 kênh** làm Miền B (`trainB`).
   - Kiểm toán tự động `Overlap(trainA, trainB) == 0` và `Overlap(train, val) == 0`. Khi lấy mẫu batch (an toàn đa tiến trình `DataLoader` trên Linux/Kaggle), ảnh RGB và ảnh IR được bốc **ngẫu nhiên độc lập** và crop/lật **độc lập**, tuyệt đối không có cặp ảnh đối ứng cùng cảnh.
2. **Đúng hợp đồng kênh `RGB (3 kênh) → IR (1 kênh)` & Tương thích 100% repo `DAOD-Duccuong`:**
   - Generator `netG` nhận `[B, 3, H, W]` $\in [-1, 1]$ và xuất trực tiếp **`[B, 1, H, W]` $\in [-1, 1]$ (1 channel duy nhất)**.
   - Discriminator `netD` phân biệt trực tiếp trên không gian ảnh nhiệt **1 kênh** (`input_nc = 1`).
   - Kiến trúc `netG` trùng khớp **100% từng tên khóa trọng số (`strict=True`)** với `models/cyclegan_generator.py` và `training/gan_bridge.py` — file `best_net_G_A.pth` / `latest_net_G_A.pth` sinh ra cắm thẳng vào `--gan-weights` của dự án mà không cần sửa code.
3. **Đánh giá so sánh toàn diện với CycleGAN theo 2 cách + Lưu Baseline JSON cho Notebook 2:**
   - **Cách 1 — Các Metrics Sinh ảnh trên IR 1 kênh:** `FID` ($\downarrow$), `KID × 10^3` ($\downarrow$), `Edge Structure CosSim` ($\uparrow$), `Thermal Contrast W1` $\text{W}_1(C_o)$ ($\downarrow$), `Person Positive Contrast` ($\uparrow$), `High-Frequency Energy Ratio` (phát hiện nhiễu giấu tin *steganography*), kèm tham chiếu `SSIM / PSNR`.
   - **Cách 2 — Đóng băng Best FCOS Detector chấm điểm Detection & Chất lượng Nhãn giả (Pseudo-Label):** Đo **COCO `mAP`, `AP50`, `AP75`, per-class `AP50` (`person`, `car`, `bicycle`)** và **chất lượng nhãn giả (`PL Precision`, `PL Recall`, `PL F1`, `Mean IoU` tại ngưỡng confidence)**.
   - **Tự động xuất `cyclegan_baseline_eval.json`:** Bạn tải/lưu file JSON này lại để nạp thẳng vào **Notebook 2 (StegoGAN)** đối chiếu 3 bên mà không cần chạy đánh giá lại CycleGAN.

In [ ]:
# ==============================================================================
# CELL 1: CẤU HÌNH TẬP TRUNG (CHỈ CẦN CHỈNH ĐƯỜNG DẪN TẠI ĐÂY)
# ==============================================================================
CONFIG = {
    # --- 1. Đường dẫn dữ liệu FLIR aligned & các Checkpoint trên Kaggle ---
    "data_root": "/kaggle/input/datasets/tungduong123qwe/flir-dataset/align",

    # Checkpoint CycleGAN hiện tại của bạn (để đánh giá baseline & xuất JSON cho Notebook 2):
    "cyclegan_weights": "/kaggle/input/datasets/bdv212/generator-rgb-ir/latest_net_G_A.pth",

    # [BẮT BUỘC ĐIỀN ĐÚNG TRƯỚC KHI RUN ALL] Đường dẫn checkpoint FCOS Detector TỐT NHẤT của bạn:
    # Hỗ trợ tự động cả 2 loại checkpoint trong repo:
    #   (a) best_model.pth từ example_flir.py (chứa dict {"metadata": ..., "student": ...})
    #   (b) best_iterXXXXXX_mapX.XXXX.pth từ training_full_d3t_v2_rtc.py (state_dict của fcos_resnet50_fpn)
    "best_fcos_checkpoint": "/kaggle/input/datasets/phamduccuong05/fcos-weight/best_model.pth",

    # Kiểm tra nghiêm ngặt mọi đường dẫn & GPU ngay ở Cell 2 (tránh tốn 1.5h train trên Kaggle rồi cuối giờ mới báo sai đường dẫn Eval!):
    "strict_preflight_check": True,

    # Thư mục lưu trọng số F-LSeSim, biểu đồ và báo cáo JSON:
    "output_dir": "/kaggle/working/flsesim_rgb2ir",

    # Nơi xuất kết quả đánh giá của CycleGAN hiện tại (tải file này về để dùng luôn ở Notebook 2!):
    "save_cyclegan_eval_json": "/kaggle/working/flsesim_rgb2ir/cyclegan_baseline_eval.json",
    # Nếu bạn đã có file JSON đánh giá CycleGAN từ lần chạy trước, điền đường dẫn vào đây (để None nếu đo mới):
    "load_existing_cyclegan_eval_json": None,

    # --- 2. Quy thức tách Disjoint Unpaired (Không ghép cặp RGB - IR) ---
    "source_fraction": 0.5,       # 50% đầu làm RGB nguồn (3ch), 50% sau làm IR đích (1ch)
    "split_seed": -1,             # -1 = giữ đúng thứ tự chuẩn data/dataloaders.py & GAN_BRIDGE_WORKFLOW.md

    # --- 3. Cấu hình huấn luyện F-LSeSim (RGB 3ch -> IR 1ch) ---
    # Với FLIR (~4.400 ảnh/miền, crop 384x384, batch_size=2): ~17-18 phút/epoch trên Kaggle T4 -> chạy 30 epochs mất ~8.5-9.5h (an toàn dưới 12h!).
    "run_training": True,         # Đặt False nếu chỉ muốn nạp checkpoint F-LSeSim có sẵn để chạy Eval
    "eval_flsesim_weights": None, # Nếu run_training=False, điền đường dẫn file .pth của F-LSeSim vào đây
    # Đường dẫn flsesim_resume_latest.pth (hỗ trợ cả file .pth lẫn thư mục bị Kaggle tự giải nén!):
    "resume_train_state": "/kaggle/input/datasets/phamduccuong05/flsesim-epoch-39/flsesim_resume_latest.pth",

    "flsesim_mode": "fsesim",     # "fsesim" (mặc định ổn định theo train_sc.sh gốc) hoặc "lsesim" (học thêm netF + InfoNCE)
    "n_epochs": 39,               # Đã chạy xong 39 epoch giữ nguyên LR ở phiên trước
    "n_epochs_decay": 30,         # Chạy tiếp đúng 30 epoch (40 -> 69) giảm tuyến tính LR về 0 (tổng = 69 epochs)
    "batch_size": 2,              # Batch size khi train GAN (~3.4 GB VRAM ở crop 384x384)
    "load_short_edge": 420,       # Resize cạnh ngắn về 420 (giữ tỷ lệ gốc 640x512 -> ~525x420)
    "crop_size": 384,             # Random crop 384x384 độc lập giữa miền RGB và miền IR
    "lr": 1e-4,                   # Chuẩn paper F-LSeSim (1e-4 với Adam beta1=0.5, beta2=0.999)
    "lambda_spatial": 10.0,       # Trọng số Spatially-Correlative Loss giữ cấu trúc (có thể hạ nhẹ 7.0 nếu muốn đẩy tương phản nhiệt mạnh hơn)
    "attn_layers": (4, 7, 9),     # Các tầng VGG16: relu3_1 (4), relu4_1 (7), relu4_3 (9)
    "patch_nums": 256,            # Số lượng vị trí neo truy vấn mỗi tầng
    "patch_size": 64,             # Kích thước vùng lân cận tính tự tương đồng
    "use_norm": True,             # Chuẩn hóa L2 đặc trưng khi tính ma trận tự tương đồng
    "temperature": 0.07,          # Nhiệt độ cho chế độ lsesim
    "pool_size": 50,              # Bộ đệm ảnh giả cho PatchGAN Discriminator
    "save_every_epochs": 10,      # Lưu mốc trọng số định kỳ mỗi N epochs
    "probe_every_epochs": 10,     # Đo FID trên tập validation mỗi N epochs để lưu best_net_G_A.pth khi FID THẤP NHẤT (0 = tắt)
    "probe_fid_max_images": None, # None = đo FID chuẩn trên TOÀN BỘ align_validation.txt (~1.013 ảnh, ~12s/lần trên GPU); đặt số nguyên nếu muốn đo tập con

    # --- 4. Cấu hình Đánh giá Kép (Generation Metrics + Frozen FCOS Detector) ---
    "eval_batch_size": 4,
    "num_workers": 2,
    "fcos_fallback_img_size": 640,     # Dùng cho checkpoint loại training_full_d3t_v2_rtc.py (mặc định resize 640x640)
    "fcos_score_thresh": 0.001,        # Ngưỡng thấp chuẩn COCO để tính chính xác đường cong Precision-Recall (mAP)
    "pseudo_label_confs": (0.3, 0.5),  # Các ngưỡng confidence để chấm trực tiếp chất lượng Nhãn giả (Pseudo-Label F1/IoU)
    "kid_subsets": 50,
    "kid_subset_size": 100,
    "eval_real_ir_upper_bound": True,  # Chấm thêm Detector trên ảnh IR thật của tập val để làm mốc trần tham chiếu
    "num_visual_samples": 6,           # Số cảnh vẽ đối chiếu trực quan ở Cell cuối

    "device": "cuda",
    "seed": 42,
}

In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, QUÉT INPUT KAGGLE & KIỂM TOÁN CHỐNG RÒ RỈ (UNPAIRED AUDIT)
# ==============================================================================
import contextlib
import copy
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
from PIL import Image
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms.functional as TF

# Tự động kiểm tra pycocotools (nếu có mạng và chưa cài thì cài nhẹ; nếu offline vẫn có bộ tính COCO AP nội bộ dự phòng)
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    HAS_PYCOCOTOOLS = True
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        HAS_PYCOCOTOOLS = True
    except Exception:
        HAS_PYCOCOTOOLS = False
        print("[INFO] Không cài được pycocotools (môi trường offline); sẽ dùng bộ tính Exact 101-Point COCO AP nội bộ.")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def dataloader_worker_init_fn(worker_id: int) -> None:
    """Đảm bảo các worker fork trên Linux/Kaggle có seed độc lập tuyệt đối ở mọi thư viện."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def atomic_torch_save(obj: Any, target_path: Path) -> None:
    """Lưu file nguyên tử qua file tạm để tránh hỏng checkpoint nếu Kaggle dừng đột ngột."""
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = target_path.with_suffix(target_path.suffix + ".tmp")
    torch.save(obj, tmp_path)
    os.replace(tmp_path, target_path)


def scan_kaggle_inputs_hint() -> None:
    """Hỗ trợ người dùng trên Kaggle: tự động liệt kê các thư mục align/ và file .pth trong /kaggle/input."""
    kaggle_in = Path("/kaggle/input")
    if not kaggle_in.is_dir():
        return
    print("\n[GỢI Ý ĐƯỜNG DẪN TÌM THẤY TRONG /kaggle/input]:")
    splits = list(kaggle_in.rglob("align_train.txt"))
    pths = list(kaggle_in.rglob("*.pth"))
    json_files = list(kaggle_in.rglob("*cyclegan*eval*.json"))
    for s in splits[:10]:
        print(f"  • data_root gợi ý           : \"{s.parent}\"")
    for p in pths[:20]:
        print(f"  • checkpoint (.pth) tìm thấy: \"{p}\"")
    for j in json_files[:10]:
        print(f"  • baseline JSON tìm thấy    : \"{j}\"")


seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if (CONFIG["device"] == "cuda" and torch.cuda.is_available()) else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLIR_CLASSES: Tuple[str, ...] = ("person", "car", "bicycle")
FLIR_CLASS_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(FLIR_CLASSES)}
IGNORE_LABELS = {"FLIR", "dog"}
IR_EXTENSIONS = (".jpeg", ".jpg", ".png")


def resolve_split_file(root: Path, split: str) -> Path:
    filename = f"align_{split}.txt"
    for candidate in (root / filename, root / "ImageSets" / "Main" / filename):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy '{filename}' trong {root}")


def read_split_stems(split_file: Path) -> List[str]:
    return [line.strip() for line in split_file.read_text(encoding="utf-8").splitlines() if line.strip()]


def ir_stem_to_rgb_path(root: Path, stem: str) -> Optional[Path]:
    base = stem.replace("_PreviewData", "")
    for ext in (".jpg", ".JPG", ".jpeg", ".png"):
        candidate = root / "JPEGImages" / f"{base}_RGB{ext}"
        if candidate.is_file():
            return candidate
    return None


def ir_stem_to_ir_path(root: Path, stem: str) -> Optional[Path]:
    for ext in IR_EXTENSIONS:
        candidate = root / "JPEGImages" / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    return None


def parse_voc_xml(xml_path: Path, min_area: float = 0.0) -> Tuple[torch.Tensor, torch.Tensor]:
    if not xml_path.is_file():
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="").strip()
        if name in IGNORE_LABELS or name not in FLIR_CLASS_TO_IDX:
            continue
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        x1 = float(bnd.findtext("xmin"))
        y1 = float(bnd.findtext("ymin"))
        x2 = float(bnd.findtext("xmax"))
        y2 = float(bnd.findtext("ymax"))
        if x2 > x1 and y2 > y1 and (x2 - x1) * (y2 - y1) >= min_area:
            boxes.append([x1, y1, x2, y2])
            labels.append(FLIR_CLASS_TO_IDX[name])
    if not boxes:
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


def partition_disjoint_stems(root: Path, source_fraction: float = 0.5, split_seed: int = -1) -> Tuple[List[str], List[str], List[str]]:
    """Tách tập train thành 2 nửa KHÔNG GIAO NHAU (khớp data/dataloaders.py)."""
    train_stems = read_split_stems(resolve_split_file(root, "train"))
    val_stems = read_split_stems(resolve_split_file(root, "validation"))
    if len(train_stems) < 2:
        raise ValueError("align_train.txt cần tối thiểu 2 mẫu để tách disjoint.")
    if len(set(train_stems)) != len(train_stems):
        raise ValueError("align_train.txt có stem trùng lặp; vi phạm điều kiện disjoint!")

    ordered = list(train_stems)
    if split_seed >= 0:
        random.Random(split_seed).shuffle(ordered)
    midpoint = int(len(ordered) // 2) if source_fraction == 0.5 else int(round(len(ordered) * source_fraction))
    rgb_source_stems = ordered[:midpoint]
    ir_target_stems = ordered[midpoint:]

    # KIỂM TOÁN CHỐNG RÒ RỈ CẶP ẢNH (HARD ASSERTIONS)
    assert set(rgb_source_stems).isdisjoint(set(ir_target_stems)), "LỖI RÒ RỈ: Tập RGB nguồn và IR đích có chung stem!"
    assert set(rgb_source_stems).isdisjoint(set(val_stems)), "LỖI RÒ RỈ: Tập RGB train trùng với validation!"
    assert set(ir_target_stems).isdisjoint(set(val_stems)), "LỖI RÒ RỈ: Tập IR train trùng với validation!"
    return rgb_source_stems, ir_target_stems, val_stems


class UnpairedFLIRRGB2IRDataset(Dataset):
    """
    Dataset huấn luyện Unpaired 100% (an toàn đa tiến trình với torch RNG):
      - Miền A (RGB): đọc đúng 3 kênh [3, H, W] trong [-1, 1] từ nửa đầu align_train.txt
      - Miền B (IR) : đọc đúng 1 kênh [1, H, W] trong [-1, 1] từ nửa sau align_train.txt (bốc ngẫu nhiên độc lập!)
      - Phép crop và lật ngang của A và B hoàn toàn độc lập nhau.
    """

    def __init__(
        self,
        root: Path,
        rgb_stems: Sequence[str],
        ir_stems: Sequence[str],
        *,
        load_short_edge: int = 420,
        crop_size: int = 384,
        use_augment_views: bool = False,
    ) -> None:
        assert set(rgb_stems).isdisjoint(set(ir_stems)), "RGB stems và IR stems bắt buộc rời nhau hoàn toàn!"
        self.rgb_paths = [p for s in rgb_stems if (p := ir_stem_to_rgb_path(root, s)) is not None]
        self.ir_paths = [p for s in ir_stems if (p := ir_stem_to_ir_path(root, s)) is not None]
        if not self.rgb_paths or not self.ir_paths:
            raise RuntimeError(f"Không tìm thấy ảnh hợp lệ: RGB={len(self.rgb_paths)}, IR={len(self.ir_paths)}")
        self.load_short_edge = int(load_short_edge)
        self.crop_size = int(crop_size)
        self.use_augment_views = bool(use_augment_views)

    def __len__(self) -> int:
        return max(len(self.rgb_paths), len(self.ir_paths))

    @staticmethod
    def _rand_uniform(low: float, high: float) -> float:
        return float(low + (high - low) * float(torch.rand((), dtype=torch.float32).item()))

    def _resize_and_independent_crop(self, pil_img: Image.Image) -> Image.Image:
        w, h = pil_img.size
        scale = max(self.load_short_edge / min(w, h), self.crop_size / min(w, h))
        new_w = max(self.crop_size, int(round(w * scale)))
        new_h = max(self.crop_size, int(round(h * scale)))
        pil_img = pil_img.resize((new_w, new_h), Image.BICUBIC)
        top = int(torch.randint(0, new_h - self.crop_size + 1, (1,)).item())
        left = int(torch.randint(0, new_w - self.crop_size + 1, (1,)).item())
        pil_img = pil_img.crop((left, top, left + self.crop_size, top + self.crop_size))
        if float(torch.rand((), dtype=torch.float32).item()) < 0.5:
            pil_img = TF.hflip(pil_img)
        return pil_img

    @staticmethod
    def _rgb_to_minus1_1(pil_rgb: Image.Image) -> torch.Tensor:
        t = TF.to_tensor(pil_rgb)  # [3, H, W] in [0, 1]
        assert t.shape[0] == 3
        return t * 2.0 - 1.0

    @staticmethod
    def _ir_1ch_to_minus1_1(pil_l: Image.Image) -> torch.Tensor:
        t = TF.to_tensor(pil_l)  # [1, H, W] in [0, 1] (vì mode='L'!)
        assert t.shape[0] == 1, f"Ảnh IR phải đúng 1 kênh, nhưng nhận được {t.shape[0]} kênh!"
        return t * 2.0 - 1.0

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        rgb_path = self.rgb_paths[idx % len(self.rgb_paths)]
        # Dùng torch.randint để đa tiến trình DataLoader trên Linux/Kaggle không bao giờ trùng chuỗi ngẫu nhiên giữa các worker
        ir_idx = int(torch.randint(0, len(self.ir_paths), (1,)).item())
        ir_path = self.ir_paths[ir_idx]

        rgb_pil = self._resize_and_independent_crop(Image.open(rgb_path).convert("RGB"))
        ir_pil = self._resize_and_independent_crop(Image.open(ir_path).convert("L"))  # BẮT BUỘC 1 CHANNEL ('L')

        real_A = self._rgb_to_minus1_1(rgb_pil)     # [3, crop, crop]
        real_B = self._ir_1ch_to_minus1_1(ir_pil)   # [1, crop, crop]

        sample = {
            "A": real_A,
            "B": real_B,
            "A_path": str(rgb_path),
            "B_path": str(ir_path),
        }
        if self.use_augment_views:
            aug_rgb = TF.adjust_hue(
                TF.adjust_saturation(
                    TF.adjust_contrast(
                        TF.adjust_brightness(rgb_pil, self._rand_uniform(0.6, 1.4)),
                        self._rand_uniform(0.6, 1.4),
                    ),
                    self._rand_uniform(0.6, 1.4),
                ),
                self._rand_uniform(-0.2, 0.2),
            )
            aug_ir = TF.adjust_contrast(
                TF.adjust_brightness(ir_pil, self._rand_uniform(0.6, 1.4)),
                self._rand_uniform(0.6, 1.4),
            )
            sample["A_aug"] = self._rgb_to_minus1_1(aug_rgb)
            sample["B_aug"] = self._ir_1ch_to_minus1_1(aug_ir)
        return sample


class FLIRValidationPairDataset(Dataset):
    """
    Chỉ dùng ở bước ĐÁNH GIÁ (align_validation.txt):
      - Cung cấp ảnh RGB [3, H, W] thuộc [0, 1] để Generator dịch sang Fake-IR 1 kênh [1, H, W].
      - Cung cấp ảnh Real-IR 1 kênh [1, H, W] thuộc [0, 1] để tính FID/KID/Contrast.
      - Cung cấp GT boxes & labels từ Annotations/{stem}.xml để chấm điểm Frozen FCOS Detector.
    """

    def __init__(self, root: Path, stems: Sequence[str], min_area: float = 0.0) -> None:
        self.root = root
        self.records = []
        for stem in stems:
            rgb_p = ir_stem_to_rgb_path(root, stem)
            ir_p = ir_stem_to_ir_path(root, stem)
            xml_p = root / "Annotations" / f"{stem}.xml"
            if rgb_p is not None and ir_p is not None and xml_p.is_file():
                self.records.append((stem, rgb_p, ir_p, xml_p))
        if not self.records:
            raise RuntimeError(f"Không tìm thấy mẫu validation hợp lệ trong {root}")
        self.min_area = min_area

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        stem, rgb_p, ir_p, xml_p = self.records[idx]
        rgb_t = TF.to_tensor(Image.open(rgb_p).convert("RGB"))  # [3, H, W] in [0, 1]
        ir_1ch = TF.to_tensor(Image.open(ir_p).convert("L"))    # [1, H, W] in [0, 1]
        boxes, labels = parse_voc_xml(xml_p, min_area=self.min_area)
        return {
            "stem": stem,
            "rgb": rgb_t,
            "ir_1ch": ir_1ch,
            "target": {"boxes": boxes, "labels": labels, "stem": stem},
        }


def val_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "stems": tuple(item["stem"] for item in batch),
        "rgb": [item["rgb"] for item in batch],
        "ir_1ch": [item["ir_1ch"] for item in batch],
        "targets": [item["target"] for item in batch],
    }


gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (CHƯA BẬT GPU!)"
print(f"[ENV] Device = {DEVICE} ({gpu_name}) | PyTorch = {torch.__version__} | pycocotools = {HAS_PYCOCOTOOLS}")

DATA_ROOT = Path(CONFIG["data_root"])
if not DATA_ROOT.is_dir():
    scan_kaggle_inputs_hint()
    if CONFIG.get("strict_preflight_check", True):
        raise FileNotFoundError(
            f"[PREFLIGHT DỪNG SỚM] Không tìm thấy data_root='{DATA_ROOT}'. "
            "Hãy copy đường dẫn gợi ý bên trên vào Cell 1 (CONFIG['data_root'])!"
        )
else:
    RGB_TRAIN_STEMS, IR_TRAIN_STEMS, VAL_STEMS = partition_disjoint_stems(
        DATA_ROOT, CONFIG["source_fraction"], CONFIG["split_seed"]
    )
    train_unpaired_ds = UnpairedFLIRRGB2IRDataset(
        DATA_ROOT,
        RGB_TRAIN_STEMS,
        IR_TRAIN_STEMS,
        load_short_edge=CONFIG["load_short_edge"],
        crop_size=CONFIG["crop_size"],
        use_augment_views=(CONFIG["flsesim_mode"].lower() == "lsesim"),
    )
    val_eval_ds = FLIRValidationPairDataset(DATA_ROOT, VAL_STEMS)
    probe_sample = train_unpaired_ds[0]
    assert probe_sample["A"].shape == (3, CONFIG["crop_size"], CONFIG["crop_size"])
    assert probe_sample["B"].shape == (1, CONFIG["crop_size"], CONFIG["crop_size"])
    print("=" * 76)
    print("[KIỂM TOÁN DỮ LIỆU UNPAIRED & KÊNH THÀNH CÔNG]")
    print(f"  • Miền A (RGB Train - 3 kênh) : {len(train_unpaired_ds.rgb_paths):4d} ảnh | Tensor shape = {tuple(probe_sample['A'].shape)}")
    print(f"  • Miền B (IR Train  - 1 kênh) : {len(train_unpaired_ds.ir_paths):4d} ảnh | Tensor shape = {tuple(probe_sample['B'].shape)}")
    print("  • Giao miền A ∩ B             : 0 ảnh (100% Unpaired / Không trùng cảnh)")
    print(f"  • Tập Validation độc lập      : {len(val_eval_ds):4d} ảnh")
    print("=" * 76)

In [ ]:
# ==============================================================================
# CELL 3: KIẾN TRÚC GENERATOR (3ch -> 1ch KHỚP 100% REPO), DISCRIMINATOR (1ch) & LOSS F-LSeSim
# ==============================================================================
from torchvision.models import VGG16_Weights, vgg16


class _ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, images: Tensor) -> Tensor:
        return images + self.conv_block(images)


class CycleGANGenerator(nn.Module):
    """
    Khớp 100% từng tham số & tên khóa với models/cyclegan_generator.py trong DAOD-Duccuong!
    Đầu vào: RGB [B, 3, H, W] trong [-1, 1]
    Đầu ra : IR  [B, 1, H, W] trong [-1, 1] (ĐÚNG 1 CHANNEL!)
    """

    def __init__(self, ngf: int = 64):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7),
            nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True),
        ]
        channels = ngf
        for _ in range(2):
            layers.extend([
                nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(channels * 2), nn.ReLU(inplace=True),
            ])
            channels *= 2
        layers.extend(_ResidualBlock(channels) for _ in range(9))
        for _ in range(2):
            layers.extend([
                nn.ConvTranspose2d(channels, channels // 2, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(channels // 2), nn.ReLU(inplace=True),
            ])
            channels //= 2
        layers.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, 1, 7), nn.Tanh()])
        self.model = nn.Sequential(*layers)

    def forward(self, images: Tensor) -> Tensor:
        return self.model(images)


class FrozenGANTranslator(nn.Module):
    """
    Khớp 100% training/gan_bridge.py:
      - translate_1ch(rgb_01) -> trả về ảnh IR 1 KÊNH [B, 1, H, W] trong [0, 1] (dùng cho metrics sinh ảnh).
      - forward(rgb_01)       -> lặp .repeat(1, 3, 1, 1) ở bước cuối cùng cho FCOS Detector.
    """

    def __init__(self, generator: nn.Module):
        super().__init__()
        self.generator = generator
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def translate_1ch(self, images_rgb_01: Tensor) -> Tensor:
        if images_rgb_01.ndim != 4 or images_rgb_01.shape[1] != 3:
            raise ValueError(f"Input phải có dạng [B, 3, H, W], nhận được {tuple(images_rgb_01.shape)}")
        height, width = images_rgb_01.shape[-2:]
        pad_h = max(8, (height + 3) // 4 * 4) - height
        pad_w = max(8, (width + 3) // 4 * 4) - width
        outputs_1ch = []
        for image in images_rgb_01.split(1):
            gan_input = F.pad(image * 2.0 - 1.0, (0, pad_w, 0, pad_h), mode="replicate")
            raw_out = self.generator(gan_input)
            if raw_out.shape != (1, 1, height + pad_h, width + pad_w):
                raise ValueError(f"Generator bắt buộc xuất 1 kênh [1,1,H,W], nhận được {tuple(raw_out.shape)}")
            thermal_1ch = ((raw_out[:, :, :height, :width] + 1.0) * 0.5).clamp(0.0, 1.0)
            outputs_1ch.append(thermal_1ch)
        return torch.cat(outputs_1ch, dim=0)  # [B, 1, H, W]

    @torch.no_grad()
    def forward(self, images_rgb_01: Tensor) -> Tensor:
        return self.translate_1ch(images_rgb_01).repeat(1, 3, 1, 1)  # [B, 3, H, W] cho FCOS


def load_gan_translator(checkpoint_path: str | Path, device: torch.device) -> FrozenGANTranslator:
    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy GAN checkpoint: {path}")
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}
    gen = CycleGANGenerator()
    gen.load_state_dict(state, strict=True)
    return FrozenGANTranslator(gen).to(device)


class PatchGANDiscriminator1Ch(nn.Module):
    """70x70 PatchGAN Discriminator nhận trực tiếp ảnh IR 1 KÊNH (input_nc = 1)."""

    def __init__(self, input_nc: int = 1, ndf: int = 64, n_layers: int = 3):
        super().__init__()
        kw, padw = 4, 1
        layers: List[nn.Module] = [
            nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        nf_mult = 1
        for n in range(1, n_layers):
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            layers.extend([
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=2, padding=padw, bias=True),
                nn.InstanceNorm2d(ndf * nf_mult),
                nn.LeakyReLU(0.2, inplace=True),
            ])
        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        layers.extend([
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=1, padding=padw, bias=True),
            nn.InstanceNorm2d(ndf * nf_mult),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * nf_mult, 1, kernel_size=kw, stride=1, padding=padw),
        ])
        self.model = nn.Sequential(*layers)

    def forward(self, x_1ch: Tensor) -> Tensor:
        return self.model(x_1ch)


def init_xavier_weights(net: nn.Module, gain: float = 0.02) -> None:
    """Khởi tạo Xavier chuẩn theo options/base_options.py của F-LSeSim."""
    def _init_fn(m: nn.Module) -> None:
        classname = m.__class__.__name__
        if hasattr(m, "weight") and m.weight is not None and ("Conv" in classname or "Linear" in classname):
            nn.init.xavier_normal_(m.weight.data, gain=gain)
            if hasattr(m, "bias") and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
    net.apply(_init_fn)


class ImagePool:
    def __init__(self, pool_size: int = 50):
        self.pool_size = pool_size
        self.images: List[Tensor] = []

    def query(self, images: Tensor) -> Tensor:
        if self.pool_size <= 0:
            return images
        out = []
        for image in images:
            image = image.detach().unsqueeze(0)
            if len(self.images) < self.pool_size:
                self.images.append(image)
                out.append(image)
            elif random.random() > 0.5:
                idx = random.randint(0, self.pool_size - 1)
                tmp = self.images[idx].clone()
                self.images[idx] = image
                out.append(tmp)
            else:
                out.append(image)
        return torch.cat(out, dim=0)


class ImageNetBroadcastNorm(nn.Module):
    """
    Chuẩn hóa ImageNet: tự động broadcast ảnh IR [B, 1, H, W] sang [B, 3, H, W]
    để đi qua bộ trích đặc trưng VGG16 mà vẫn giữ nguyên đạo hàm về 1 kênh của Generator!
    """

    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, img_01: Tensor) -> Tensor:
        return (img_01 - self.mean) / self.std


class VGG16FeatureExtractor(nn.Module):
    """Khớp chính xác chỉ số tầng VGG16 của lyndonzheng/F-LSeSim (4=relu3_1, 7=relu4_1, 9=relu4_3)."""

    BLOCK_RANGES = (
        (0, 2), (2, 4),          # 0: relu1_1, 1: relu1_2
        (4, 7), (7, 9),          # 2: relu2_1, 3: relu2_2
        (9, 12), (12, 14), (14, 16),   # 4: relu3_1, 5: relu3_2, 6: relu3_3
        (16, 18), (18, 21), (21, 23),  # 7: relu4_1, 8: relu4_2, 9: relu4_3
        (23, 26), (26, 28), (28, 30),  # 10: relu5_1, 11: relu5_2, 12: relu5_3
    )

    def __init__(self, weights_mode: str = "pretrained"):
        super().__init__()
        try:
            vgg_feats = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if weights_mode == "pretrained" else None).features
        except Exception as err:
            raise RuntimeError(
                "Không tải được trọng số VGG16 ImageNet (hãy kiểm tra đã bật 'Internet: On' trong Settings của Kaggle Notebook!)."
            ) from err
        self.blocks = nn.ModuleList([
            nn.Sequential(*[vgg_feats[i] for i in range(start, end)])
            for start, end in self.BLOCK_RANGES
        ])
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    def forward(self, x_norm_3ch: Tensor, layers: Sequence[int] = (4, 7, 9)) -> List[Tensor]:
        max_layer = max(layers)
        target_set = set(layers)
        feats: List[Tensor] = []
        h = x_norm_3ch
        for idx, block in enumerate(self.blocks):
            h = block(h)
            if idx in target_set:
                feats.append(h)
            if idx >= max_layer:
                break
        return feats


class SpatialCorrelativeLoss(nn.Module):
    """
    Cài đặt chính xác Spatially-Correlative Loss (Zheng et al., CVPR 2021):
    So sánh bản đồ TỰ TƯƠNG ĐỒNG NỘI TẠI S_x vs S_y -> bất biến hoàn toàn với đảo cực sáng-tối RGB -> IR!
    Tối ưu tính toán theo chunk truy vấn để tiết kiệm 75% VRAM ở độ phân giải cao.
    """

    def __init__(
        self,
        *,
        patch_nums: int = 256,
        patch_size: int = 64,
        use_norm: bool = True,
        learned_attn: bool = False,
        temperature: float = 0.07,
        chunk_size: int = 64,
    ):
        super().__init__()
        self.patch_nums = int(patch_nums)
        self.patch_size = int(patch_size)
        self.use_norm = bool(use_norm)
        self.learned_attn = bool(learned_attn)
        self.temperature = float(temperature)
        self.chunk_size = int(chunk_size)
        self.projs = nn.ModuleDict()
        self.l1 = nn.L1Loss() if self.use_norm else nn.SmoothL1Loss()
        self.ce = nn.CrossEntropyLoss()

    def _get_or_build_proj(self, feat: Tensor, layer_idx: int) -> nn.Module:
        key = f"l_{layer_idx}"
        if key not in self.projs:
            in_c = feat.shape[1]
            out_c = max(32, in_c // 4)
            proj = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, kernel_size=1),
            ).to(feat.device)
            init_xavier_weights(proj, gain=0.02)
            self.projs[key] = proj
        return self.projs[key]

    def _compute_patch_self_similarity(
        self, feat: Tensor, patch_ids: Optional[Tensor] = None
    ) -> Tuple[Tensor, Tensor]:
        B, C, H, W = feat.shape
        feat_norm = F.normalize(feat, dim=1) if self.use_norm else (feat / math.sqrt(C))
        total_locs = H * W
        if patch_ids is None:
            num_q = min(self.patch_nums, total_locs) if self.patch_nums > 0 else total_locs
            patch_ids = torch.randperm(total_locs, device=feat.device)[:num_q]

        num_q = patch_ids.numel()
        pw, ph = self.patch_size, self.patch_size
        flat = feat_norm.flatten(2)  # [B, C, H*W]
        queries_all = flat[:, :, patch_ids].permute(0, 2, 1)  # [B, num_q, C]

        if pw < W and ph < H:
            rows = patch_ids // W
            cols = patch_ids % W
            top = (rows - ph // 2).clamp(0, H - ph)
            left = (cols - pw // 2).clamp(0, W - pw)
            sim_chunks = []
            for start in range(0, num_q, self.chunk_size):
                end = min(start + self.chunk_size, num_q)
                q_chunk = queries_all[:, start:end, :]  # [B, K, C]
                keys = torch.stack(
                    [
                        feat_norm[:, :, int(top[i]):int(top[i]) + ph, int(left[i]):int(left[i]) + pw].flatten(2)
                        for i in range(start, end)
                    ],
                    dim=1,
                )  # [B, K, C, ph*pw]
                K_cur = end - start
                sim_c = torch.bmm(
                    q_chunk.reshape(B * K_cur, 1, C),
                    keys.reshape(B * K_cur, C, ph * pw),
                ).reshape(B, K_cur, ph * pw)
                sim_chunks.append(sim_c if self.use_norm else torch.tanh(sim_c / 10.0))
            patch_sim = torch.cat(sim_chunks, dim=1)
        else:
            sim = torch.bmm(queries_all, flat)  # [B, num_q, H*W]
            patch_sim = sim if self.use_norm else torch.tanh(sim / 10.0)
        return patch_sim, patch_ids

    def layer_loss(
        self, f_src: Tensor, f_tgt: Tensor, layer_idx: int, f_other: Optional[Tensor] = None
    ) -> Tensor:
        if self.learned_attn:
            proj = self._get_or_build_proj(f_src, layer_idx)
            f_src = proj(f_src)
            f_tgt = proj(f_tgt)
            if f_other is not None:
                f_other = proj(f_other)

        sim_src, patch_ids = self._compute_patch_self_similarity(f_src, None)
        sim_tgt, _ = self._compute_patch_self_similarity(f_tgt, patch_ids)

        if f_other is not None:
            sim_oth, _ = self._compute_patch_self_similarity(f_other, patch_ids)
            B, Num, _ = sim_src.shape
            s_src = F.normalize(sim_src, dim=-1)
            s_tgt = F.normalize(sim_tgt, dim=-1)
            s_oth = F.normalize(sim_oth, dim=-1)
            neg1 = torch.bmm(s_src, s_oth.transpose(1, 2)).reshape(-1, Num) / self.temperature
            neg2 = torch.bmm(s_tgt, s_oth.transpose(1, 2)).reshape(-1, Num) / self.temperature
            pos = torch.bmm(s_src, s_tgt.transpose(1, 2)).reshape(-1, Num) / self.temperature
            logits = torch.cat([pos, neg1, neg2], dim=-1)
            labels = torch.arange(logits.size(0), device=logits.device, dtype=torch.long) % Num
            return self.ce(logits, labels)

        # Chế độ FSeSim chuẩn (so khớp cấu trúc top-25% vùng lân cận tương đồng nhất bằng Cosine distance)
        N = sim_tgt.shape[-1]
        keep = max(1, N // 4)
        tgt_sorted, _ = sim_tgt.sort(dim=-1, descending=True)
        thresh = tgt_sorted[:, :, keep - 1 : keep]
        src_masked = torch.where(sim_tgt < thresh, torch.zeros_like(sim_src), sim_src)
        tgt_masked = torch.where(sim_tgt < thresh, torch.zeros_like(sim_tgt), sim_tgt)
        cos_pos = F.cosine_similarity(src_masked, tgt_masked, dim=-1)
        return self.l1(torch.ones_like(cos_pos), cos_pos)

    def forward(
        self,
        feats_src: Sequence[Tensor],
        feats_tgt: Sequence[Tensor],
        feats_other: Optional[Sequence[Tensor]] = None,
    ) -> Tensor:
        total = torch.zeros((), device=feats_src[0].device)
        for idx, (fs, ft) in enumerate(zip(feats_src, feats_tgt)):
            fo = None if feats_other is None else feats_other[idx]
            total = total + self.layer_loss(fs, ft, idx, fo)
        return total / max(1, len(feats_src))

In [ ]:
# ==============================================================================
# CELL 4: BỘ NẠP ĐÓNG BĂNG BEST FCOS DETECTOR, PREFLIGHT CHECK & 2 BỘ ĐÁNH GIÁ
# ==============================================================================
from collections import OrderedDict
from torchvision.models import Inception_V3_Weights, inception_v3
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSClassificationHead, FCOSHead


# --- 4.1 Hỗ trợ tự động CẢ HAI định dạng Checkpoint FCOS của dự án DAOD-Duccuong ---
class _RegressionHeadWithoutCenterness(nn.Module):
    def __init__(self, conv: nn.Module, bbox_reg: nn.Module, num_anchors: int):
        super().__init__()
        self.conv = conv
        self.bbox_reg = bbox_reg
        self.num_anchors = num_anchors

    def forward(self, features: Sequence[Tensor]) -> Tensor:
        outputs = []
        for feature in features:
            reg = F.relu(self.bbox_reg(self.conv(feature)))
            b, _, h, w = reg.shape
            outputs.append(reg.view(b, self.num_anchors, 4, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, 4))
        return torch.cat(outputs, dim=1)


class _StandaloneFCOSIoUHead(nn.Module):
    """Tái tạo chính xác models/fcos_adapter.py (FCOSIoUHead) để nạp best_model.pth từ example_flir.py."""

    def __init__(self, base_head: FCOSHead, num_classes: int = 3):
        super().__init__()
        base_cls = base_head.classification_head
        base_reg = base_head.regression_head
        in_channels = next(m.in_channels for m in base_cls.conv if isinstance(m, nn.Conv2d))
        num_anchors = base_cls.num_anchors
        cls_head = copy.deepcopy(base_cls)
        cls_head.num_classes = num_classes
        cls_head.cls_logits = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=3, stride=1, padding=1)

        self.classification_head = cls_head
        self.regression_head = _RegressionHeadWithoutCenterness(
            conv=copy.deepcopy(base_reg.conv),
            bbox_reg=copy.deepcopy(base_reg.bbox_reg),
            num_anchors=num_anchors,
        )
        self.quality_predictor = nn.Conv2d(in_channels, num_anchors, kernel_size=3, stride=1, padding=1)
        self.num_classes = num_classes

    @staticmethod
    def _flatten(out: Tensor, c_per_anchor: int) -> Tensor:
        b, _, h, w = out.shape
        return out.view(b, -1, c_per_anchor, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, c_per_anchor)

    def forward(self, features: Sequence[Tensor]) -> Dict[str, Tensor]:
        cls_outs, qual_outs = [], []
        for f in features:
            cf = self.classification_head.conv(f)
            cls_outs.append(self._flatten(self.classification_head.cls_logits(cf), self.num_classes))
            qual_outs.append(self._flatten(self.quality_predictor(cf), 1))
        return {
            "cls_logits": torch.cat(cls_outs, dim=1),
            "bbox_regression": self.regression_head(features),
            "quality_logits": torch.cat(qual_outs, dim=1),
        }


class _StandaloneIoUFCOSInference(nn.Module):
    """Chạy inference chính xác giống TorchvisionFCOSAdapter.postprocess()."""

    def __init__(self, detector: nn.Module):
        super().__init__()
        self.detector = detector

    @torch.no_grad()
    def forward(self, images: List[Tensor]) -> List[Dict[str, Tensor]]:
        orig_sizes = [(int(img.shape[-2]), int(img.shape[-1])) for img in images]
        transformed, _ = self.detector.transform(images, None)
        feats = self.detector.backbone(transformed.tensors)
        if isinstance(feats, Tensor):
            feats = OrderedDict([("0", feats)])
        feat_list = list(feats.values())
        head_out = self.detector.head(feat_list)
        anchors_list = self.detector.anchor_generator(transformed, feat_list)
        anchors_per_loc = self.detector.anchor_generator.num_anchors_per_location()
        num_anchors_per_level = [
            int(f.shape[-2] * f.shape[-1] * anchors_per_loc[lvl])
            for lvl, f in enumerate(feat_list)
        ]
        anchors_per_level = [list(a.split(num_anchors_per_level, dim=0)) for a in anchors_list]
        per_level = {k: list(v.split(num_anchors_per_level, dim=1)) for k, v in head_out.items()}
        native = self.detector.postprocess_detections(
            {
                "cls_logits": per_level["cls_logits"],
                "bbox_regression": per_level["bbox_regression"],
                "bbox_ctrness": per_level["quality_logits"],
            },
            anchors_per_level,
            list(transformed.image_sizes),
        )
        return self.detector.transform.postprocess(native, list(transformed.image_sizes), orig_sizes)


class FrozenFCOSJudge(nn.Module):
    """Giám khảo FCOS ĐÓNG BĂNG thống nhất cho cả 2 định dạng checkpoint của dự án."""

    def __init__(self, predictor: nn.Module, fmt_name: str, fixed_resize: Optional[int] = None):
        super().__init__()
        self.predictor = predictor
        self.fmt_name = fmt_name
        self.fixed_resize = fixed_resize
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def predict(self, images_3ch_01: List[Tensor]) -> List[Dict[str, Tensor]]:
        if self.fixed_resize is not None:
            resized_imgs = []
            scales = []
            for img in images_3ch_01:
                h, w = img.shape[-2:]
                scales.append((w / self.fixed_resize, h / self.fixed_resize))
                resized_imgs.append(
                    F.interpolate(img.unsqueeze(0), size=(self.fixed_resize, self.fixed_resize), mode="bilinear", align_corners=False).squeeze(0)
                )
            preds = self.predictor(resized_imgs)
            restored = []
            for pred, (sx, sy) in zip(preds, scales):
                boxes = pred["boxes"].clone()
                if boxes.numel() > 0:
                    boxes[:, [0, 2]] *= sx
                    boxes[:, [1, 3]] *= sy
                restored.append({"boxes": boxes, "scores": pred["scores"], "labels": pred["labels"]})
            return restored
        return self.predictor(images_3ch_01)


def load_frozen_fcos_detector(
    checkpoint_path: str | Path,
    device: torch.device,
    *,
    score_thresh: float = 0.001,
    fallback_img_size: int = 640,
) -> FrozenFCOSJudge:
    import pickle
    import types

    path = Path(checkpoint_path)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy best_fcos_checkpoint tại: {path}")

    class _FallbackUnpickledObj:
        """Placeholder an toàn thay cho dataclass/Enum của repo (vd: configs.types.Phase) khi chạy độc lập trên Kaggle."""
        def __new__(cls, *args, **kwargs):
            inst = super().__new__(cls)
            inst._args = args
            return inst

        def __init__(self, *args, **kwargs):
            self._args = args
            self.__dict__.update(kwargs)

        def __setstate__(self, state):
            if isinstance(state, dict):
                self.__dict__.update(state)
            else:
                self._state = state

        def get(self, key, default=None):
            return self.__dict__.get(key, default)

        @property
        def value(self):
            return self._args[0] if getattr(self, "_args", ()) else None

        def __hash__(self):
            return hash((type(self).__name__, getattr(self, "_args", ())))

        def __eq__(self, other):
            return isinstance(other, _FallbackUnpickledObj) and getattr(self, "_args", ()) == getattr(other, "_args", ())

    class _SafeUnpickler(pickle.Unpickler):
        def find_class(self, mod_name: str, name: str):
            if mod_name == "pathlib" and name in ("PosixPath", "WindowsPath"):
                return Path
            try:
                return super().find_class(mod_name, name)
            except (ModuleNotFoundError, AttributeError):
                return type(f"_Missing_{name}", (_FallbackUnpickledObj,), {"__module__": mod_name})

    safe_pickle = types.SimpleNamespace(**{k: getattr(pickle, k) for k in dir(pickle)})
    safe_pickle.Unpickler = _SafeUnpickler
    raw = torch.load(path, map_location="cpu", weights_only=False, pickle_module=safe_pickle)

    # Định dạng A: Checkpoint từ example_flir.py / trainer.py (có khóa "student" + "metadata" / adapter.detector...)
    if isinstance(raw, dict) and "student" in raw and any(k.startswith("adapter.detector.") for k in raw["student"]):
        meta = raw.get("metadata", {})
        if not isinstance(meta, dict):
            meta = vars(meta) if hasattr(meta, "__dict__") else {}
        saved_m = meta.get("model")
        if saved_m is None:
            cfg_obj = meta.get("config", {})
            if not isinstance(cfg_obj, dict):
                cfg_obj = vars(cfg_obj) if hasattr(cfg_obj, "__dict__") else {}
            saved_m = cfg_obj.get("model", {})
        if not isinstance(saved_m, dict):
            saved_m = vars(saved_m) if hasattr(saved_m, "__dict__") else {}

        num_classes = int(saved_m.get("num_classes", 3))
        raw_min = saved_m.get("min_sizes", (800,))
        min_sizes = (int(raw_min),) if isinstance(raw_min, (int, float)) else tuple(int(v) for v in raw_min)
        max_size = int(saved_m.get("max_size", 1333))
        nms_thresh = float(saved_m.get("nms_thresh", 0.6))
        topk = int(saved_m.get("topk_candidates", 1000))

        base = fcos_resnet50_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=91,
            min_size=(min_sizes[-1],),
            max_size=max_size,
            score_thresh=score_thresh,
            nms_thresh=nms_thresh,
            topk_candidates=topk,
            detections_per_img=100,
        )
        base.head = _StandaloneFCOSIoUHead(base.head, num_classes=num_classes)
        det_state = {
            k.removeprefix("adapter.detector."): v
            for k, v in raw["student"].items()
            if k.startswith("adapter.detector.")
        }
        base.load_state_dict(det_state, strict=True)
        judge = FrozenFCOSJudge(_StandaloneIoUFCOSInference(base), fmt_name="DAOD-Adapter-FCOSIoUHead", fixed_resize=None)
        return judge.to(device)

    # Định dạng B: Checkpoint từ training_full_d3t_v2_rtc.py (torchvision fcos_resnet50_fpn trực tiếp)
    state = raw
    for key in ("model", "s_model", "state_dict", "student"):
        if isinstance(state, dict) and key in state and isinstance(state[key], dict):
            state = state[key]
            break
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}

    base_std = fcos_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=len(FLIR_CLASSES),
        min_size=fallback_img_size,
        max_size=fallback_img_size,
        score_thresh=score_thresh,
        detections_per_img=100,
    )
    base_std.load_state_dict(state, strict=True)
    judge = FrozenFCOSJudge(base_std, fmt_name=f"Torchvision-FCOS-{fallback_img_size}", fixed_resize=fallback_img_size)
    return judge.to(device)


# --- 4.2 Bộ tính CÁCH EVAL 2: COCO mAP + Chất lượng Nhãn giả (Pseudo-Label Precision/Recall/F1/IoU) ---
def box_iou_matrix(boxes1: Tensor, boxes2: Tensor) -> Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp_min(0) * (boxes1[:, 3] - boxes1[:, 1]).clamp_min(0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp_min(0) * (boxes2[:, 3] - boxes2[:, 1]).clamp_min(0)
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp_min(0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    return inter / (area1[:, None] + area2[None, :] - inter).clamp_min(1e-7)


def compute_pseudo_label_quality(
    predictions: Sequence[Dict[str, Tensor]],
    targets: Sequence[Dict[str, Tensor]],
    conf_thresh: float = 0.5,
    iou_thresh: float = 0.5,
) -> Dict[str, float]:
    """Đo trực tiếp chất lượng NHÃN GIẢ (Pseudo-Label) do best FCOS sinh ra trên miền ảnh giả tại ngưỡng conf_thresh."""
    tp, fp, total_gt = 0, 0, 0
    matched_ious: List[float] = []
    for pred, tgt in zip(predictions, targets):
        gt_boxes = tgt["boxes"].cpu()
        gt_labels = tgt["labels"].cpu()
        total_gt += int(gt_boxes.shape[0])

        keep = pred["scores"].cpu() >= conf_thresh
        p_boxes = pred["boxes"].cpu()[keep]
        p_scores = pred["scores"].cpu()[keep]
        p_labels = pred["labels"].cpu()[keep]
        if p_boxes.shape[0] > 0:
            order = torch.argsort(p_scores, descending=True)
            p_boxes, p_labels = p_boxes[order], p_labels[order]

        gt_used = torch.zeros((gt_boxes.shape[0],), dtype=torch.bool)
        ious = box_iou_matrix(p_boxes, gt_boxes)
        for i in range(p_boxes.shape[0]):
            same_cls = (gt_labels == p_labels[i]) & (~gt_used)
            if bool(same_cls.any()):
                cand_ious = ious[i].clone()
                cand_ious[~same_cls] = -1.0
                best_iou, best_j = cand_ious.max(dim=0)
                if float(best_iou) >= iou_thresh:
                    tp += 1
                    gt_used[best_j] = True
                    matched_ious.append(float(best_iou))
                else:
                    fp += 1
            else:
                fp += 1

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, total_gt)
    f1 = 2.0 * precision * recall / max(1e-8, precision + recall)
    return {
        "conf_thresh": conf_thresh,
        "PL_Precision": float(precision),
        "PL_Recall": float(recall),
        "PL_F1": float(f1),
        "PL_MeanMatchedIoU": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "PL_BoxesPerImg": float((tp + fp) / max(1, len(targets))),
        "TP": int(tp),
        "FP": int(fp),
        "GT": int(total_gt),
    }


def _fallback_coco_ap(predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]) -> Dict[str, float]:
    """Bộ tính COCO 101-point AP nội bộ phòng khi chạy offline không có pycocotools."""
    iou_thrs = np.linspace(0.5, 0.95, 10)
    recall_thrs = np.linspace(0.0, 1.0, 101)
    ap_matrix = np.full((len(iou_thrs), len(FLIR_CLASSES)), np.nan, dtype=np.float64)

    for c_idx in range(len(FLIR_CLASSES)):
        gt_by_img = {}
        total_gt = 0
        det_records = []
        for img_id, (pred, tgt) in enumerate(zip(predictions, targets)):
            g_mask = tgt["labels"].cpu() == c_idx
            g_boxes = tgt["boxes"].cpu()[g_mask]
            gt_by_img[img_id] = g_boxes
            total_gt += int(g_boxes.shape[0])

            p_mask = pred["labels"].cpu() == c_idx
            p_boxes = pred["boxes"].cpu()[p_mask]
            p_scores = pred["scores"].cpu()[p_mask]
            for b, s in zip(p_boxes, p_scores):
                det_records.append((float(s), img_id, b.unsqueeze(0)))

        if total_gt == 0:
            continue
        det_records.sort(key=lambda x: x[0], reverse=True)
        for t_idx, thr in enumerate(iou_thrs):
            used = {img_id: np.zeros(len(boxes), dtype=bool) for img_id, boxes in gt_by_img.items()}
            tps = np.zeros(len(det_records), dtype=np.float64)
            fps = np.zeros(len(det_records), dtype=np.float64)
            for d_i, (_, img_id, box_1) in enumerate(det_records):
                g_boxes = gt_by_img[img_id]
                if len(g_boxes) == 0:
                    fps[d_i] = 1.0
                    continue
                ious = box_iou_matrix(box_1, g_boxes)[0].numpy()
                ious[used[img_id]] = -1.0
                j = int(np.argmax(ious))
                if ious[j] >= thr:
                    tps[d_i] = 1.0
                    used[img_id][j] = True
                else:
                    fps[d_i] = 1.0
            tp_cum = np.cumsum(tps)
            fp_cum = np.cumsum(fps)
            rec = tp_cum / total_gt
            prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
            for i in range(len(prec) - 2, -1, -1):
                prec[i] = max(prec[i], prec[i + 1])
            inds = np.searchsorted(rec, recall_thrs, side="left")
            q = np.array([prec[idx] if idx < len(prec) else 0.0 for idx in inds])
            ap_matrix[t_idx, c_idx] = float(np.mean(q))

    out = {
        "AP": float(np.nanmean(ap_matrix)),
        "AP50": float(np.nanmean(ap_matrix[0])),
        "AP75": float(np.nanmean(ap_matrix[5])),
    }
    for c_idx, name in enumerate(FLIR_CLASSES):
        out[f"AP/{name}"] = float(np.nanmean(ap_matrix[:, c_idx]))
        out[f"AP50/{name}"] = float(ap_matrix[0, c_idx])
        out[f"AP75/{name}"] = float(ap_matrix[5, c_idx])
    return out


def compute_coco_detection_metrics(
    predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]
) -> Dict[str, float]:
    if not HAS_PYCOCOTOOLS:
        return _fallback_coco_ap(predictions, targets)

    categories = [{"id": i + 1, "name": name} for i, name in enumerate(FLIR_CLASSES)]
    images = [{"id": i + 1} for i in range(len(targets))]
    annotations, detections = [], []
    ann_id = 1
    for img_idx, (pred, tgt) in enumerate(zip(predictions, targets), start=1):
        for box, lbl in zip(tgt["boxes"].cpu().tolist(), tgt["labels"].cpu().tolist()):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            annotations.append({
                "id": ann_id, "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, w, h], "area": w * h, "iscrowd": 0,
            })
            ann_id += 1
        for box, score, lbl in zip(
            pred["boxes"].cpu().tolist(), pred["scores"].cpu().tolist(), pred["labels"].cpu().tolist()
        ):
            x1, y1, x2, y2 = box
            detections.append({
                "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score),
            })

    # Tắt log thừa của pycocotools để log Kaggle gọn sạch
    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO()
        coco_gt.dataset = {"info": {}, "licenses": [], "images": images, "annotations": annotations, "categories": categories}
        coco_gt.createIndex()
        if detections:
            coco_dt = coco_gt.loadRes(detections)
        else:
            coco_dt = COCO()
            coco_dt.dataset = {"images": images, "categories": categories, "annotations": []}
            coco_dt.createIndex()

        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.params.catIds = [1, 2, 3]
        coco_eval.params.imgIds = list(range(1, len(targets) + 1))
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

    stat_names = ("AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR1", "AR10", "AR100")
    metrics = {name: float(coco_eval.stats[i]) for i, name in enumerate(stat_names)}
    precision = coco_eval.eval["precision"]
    area_idx = list(coco_eval.params.areaRngLbl).index("all")
    max_det_idx = len(coco_eval.params.maxDets) - 1
    iou_thrs = np.asarray(coco_eval.params.iouThrs)
    for c_idx, c_name in enumerate(FLIR_CLASSES):
        cp = precision[:, :, c_idx, area_idx, max_det_idx]
        valid = cp[cp > -1]
        metrics[f"AP/{c_name}"] = float(valid.mean()) if valid.size else float("nan")
        for thr, tag in ((0.5, "AP50"), (0.75, "AP75")):
            m_idx = np.flatnonzero(np.isclose(iou_thrs, thr))
            v = cp[m_idx[0]][cp[m_idx[0]] > -1]
            metrics[f"{tag}/{c_name}"] = float(v.mean()) if v.size else float("nan")
    return metrics


# --- 4.3 Bộ tính CÁCH EVAL 1: Image Generation Metrics trên ảnh IR 1 KÊNH ---
class InceptionPool3Extractor(nn.Module):
    """Trích vector 2048-D chuẩn từ InceptionV3 (pool3) để tính FID và KID cho ảnh sinh 1 kênh."""

    def __init__(self):
        super().__init__()
        try:
            net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
        except Exception as err:
            raise RuntimeError(
                "Không tải được InceptionV3 ImageNet weights (hãy kiểm tra đã bật 'Internet: On' trên Kaggle!)."
            ) from err
        net.fc = nn.Identity()
        net.aux_logits = False
        self.net = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, ir_1ch_01: Tensor) -> Tensor:
        assert ir_1ch_01.shape[1] == 1, "Inception feature extractor yêu cầu đầu vào ảnh IR đúng 1 kênh!"
        x = F.interpolate(ir_1ch_01.repeat(1, 3, 1, 1), size=(299, 299), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return self.net(x)


def compute_fid_and_kid(
    real_feats: np.ndarray,
    fake_feats: np.ndarray,
    *,
    kid_subsets: int = 50,
    kid_subset_size: int = 100,
    seed: int = 42,
) -> Dict[str, float]:
    """
    Tính FID (bằng phân rã trị riêng đối xứng PSD chuẩn float64, ổn định tuyệt đối kể cả khi N < 2048)
    và KID × 10^3 không chệch (polynomial MMD^2 bậc 3).
    """
    r64 = torch.from_numpy(np.asarray(real_feats, dtype=np.float64))
    f64 = torch.from_numpy(np.asarray(fake_feats, dtype=np.float64))

    mu1, mu2 = r64.mean(dim=0), f64.mean(dim=0)
    r_c, f_c = r64 - mu1, f64 - mu2
    sigma1 = (r_c.T @ r_c) / max(1, r64.shape[0] - 1)
    sigma2 = (f_c.T @ f_c) / max(1, f64.shape[0] - 1)

    eps_eye = torch.eye(sigma1.shape[0], dtype=torch.float64) * 1e-6
    eigvals1, eigvecs1 = torch.linalg.eigh(sigma1 + eps_eye)
    sqrt_sigma1 = (eigvecs1 * torch.sqrt(eigvals1.clamp_min(0.0))) @ eigvecs1.T
    prod_sym = sqrt_sigma1 @ (sigma2 + eps_eye) @ sqrt_sigma1
    prod_sym = 0.5 * (prod_sym + prod_sym.T)
    trace_covmean = float(torch.sqrt(torch.linalg.eigvalsh(prod_sym).clamp_min(0.0)).sum().item())
    diff = mu1 - mu2
    fid = float((diff.dot(diff) + torch.trace(sigma1) + torch.trace(sigma2) - 2.0 * trace_covmean).clamp_min(0.0).item())

    if kid_subsets <= 0:
        return {"FID": fid, "KID_x1000_mean": float("nan"), "KID_x1000_std": float("nan")}

    r_np = r64.numpy()
    f_np = f64.numpy()
    rng = np.random.default_rng(seed)
    m = min(kid_subset_size, len(r_np), len(f_np))
    d = float(r_np.shape[1])
    mmds = []
    for _ in range(kid_subsets):
        x = r_np[rng.choice(len(r_np), m, replace=False)]
        y = f_np[rng.choice(len(f_np), m, replace=False)]
        k_xx = (x @ x.T / d + 1.0) ** 3
        k_yy = (y @ y.T / d + 1.0) ** 3
        k_xy = (x @ y.T / d + 1.0) ** 3
        mmd = (
            (k_xx.sum() - np.trace(k_xx)) / max(1, m * (m - 1))
            + (k_yy.sum() - np.trace(k_yy)) / max(1, m * (m - 1))
            - 2.0 * k_xy.mean()
        )
        mmds.append(float(mmd))
    return {
        "FID": fid,
        "KID_x1000_mean": float(np.mean(mmds) * 1000.0),
        "KID_x1000_std": float(np.std(mmds) * 1000.0),
    }


def _sobel_magnitude(img_1ch: Tensor) -> Tensor:
    kx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    ky = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    gx = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), kx)
    gy = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), ky)
    return torch.sqrt(gx * gx + gy * gy + 1e-8)


def _laplacian_energy(img_1ch: Tensor) -> float:
    klap = torch.tensor([[0.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 0.0]], device=img_1ch.device).view(1, 1, 3, 3)
    resp = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), klap)
    return float((resp ** 2).mean().item())


def _ssim_1ch(x: Tensor, y: Tensor) -> float:
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mu_x = F.avg_pool2d(x, 11, 1, 5)
    mu_y = F.avg_pool2d(y, 11, 1, 5)
    sigma_x = F.avg_pool2d(x * x, 11, 1, 5) - mu_x ** 2
    sigma_y = F.avg_pool2d(y * y, 11, 1, 5) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 11, 1, 5) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2))
    return float(ssim_map.mean().item())


def _signed_box_ring_contrasts(img_1ch_2d: Tensor, boxes: Tensor, labels: Tensor) -> List[Tuple[int, float]]:
    """Tính độ tương phản nhiệt có dấu C_o giữa bên trong GT box và vành nền xung quanh."""
    H, W = img_1ch_2d.shape
    results = []
    for box, lbl in zip(boxes.tolist(), labels.tolist()):
        x1, y1, x2, y2 = box
        cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
        bw, bh = x2 - x1, y2 - y1
        if bw < 6 or bh < 6:
            continue
        ix1, iy1 = max(0, int(round(cx - 0.4 * bw))), max(0, int(round(cy - 0.4 * bh)))
        ix2, iy2 = min(W, int(round(cx + 0.4 * bw))), min(H, int(round(cy + 0.4 * bh)))
        ox1, oy1 = max(0, int(round(cx - 0.65 * bw))), max(0, int(round(cy - 0.65 * bh)))
        ox2, oy2 = min(W, int(round(cx + 0.65 * bw))), min(H, int(round(cy + 0.65 * bh)))
        if ix2 <= ix1 + 1 or iy2 <= iy1 + 1:
            continue
        core = img_1ch_2d[iy1:iy2, ix1:ix2].reshape(-1)
        outer_patch = img_1ch_2d[oy1:oy2, ox1:ox2]
        mask = torch.ones_like(outer_patch, dtype=torch.bool)
        mask[iy1 - oy1 : iy2 - oy1, ix1 - ox1 : ix2 - ox1] = False
        ring = outer_patch[mask]
        if core.numel() < 4 or ring.numel() < 8:
            continue
        med_obj = float(core.median().item())
        med_bg = float(ring.median().item())
        mad_bg = float((ring - med_bg).abs().median().item())
        c_val = (med_obj - med_bg) / (1.4826 * mad_bg + 1e-4)
        results.append((int(lbl), float(np.clip(c_val, -15.0, 15.0))))
    return results


def _wasserstein_1d(u: Sequence[float], v: Sequence[float]) -> float:
    if not u or not v:
        return float("nan")
    u_s = np.sort(np.asarray(u, dtype=np.float64))
    v_s = np.sort(np.asarray(v, dtype=np.float64))
    grid = np.linspace(0.0, 1.0, 256)
    return float(np.mean(np.abs(np.quantile(u_s, grid) - np.quantile(v_s, grid))))


@torch.no_grad()
def evaluate_generator_comprehensive(
    *,
    method_name: str,
    translator: Optional[FrozenGANTranslator],
    val_loader: DataLoader,
    device: torch.device,
    inception_extractor: Optional[InceptionPool3Extractor],
    fcos_judge: Optional[FrozenFCOSJudge],
    pseudo_confs: Sequence[float] = (0.3, 0.5),
    kid_subsets: int = 50,
    kid_subset_size: int = 100,
    is_real_ir_anchor: bool = False,
) -> Dict[str, Any]:
    """
    Chạy trọn gói CẢ 2 CÁCH ĐÁNH GIÁ trong một lượt duyệt tập Validation duy nhất:
      - Cách 1: FID, KID, Edge-Structure Cosine Sim, Thermal Contrast W1, High-Freq Steganography Ratio, SSIM/PSNR.
      - Cách 2: Đóng băng Best FCOS Detector -> COCO mAP/AP50/AP75 + Per-class AP50 + Pseudo-Label Precision/Recall/F1/IoU.
    """
    real_inc_list, fake_inc_list = [], []
    edge_cos_scores, ssim_scores, psnr_scores = [], [], []
    real_hf_energies, fake_hf_energies = [], []
    real_contrasts_all, fake_contrasts_all = [], []
    fake_person_contrasts = []

    fcos_predictions: List[Dict[str, Tensor]] = []
    fcos_targets: List[Dict[str, Tensor]] = []

    t0 = time.time()
    for batch in val_loader:
        rgb_list: List[Tensor] = batch["rgb"]
        ir_1ch_list: List[Tensor] = batch["ir_1ch"]
        targets: List[Dict[str, Tensor]] = batch["targets"]

        fcos_batch_3ch: List[Tensor] = []
        for rgb_cpu, real_ir_cpu, tgt in zip(rgb_list, ir_1ch_list, targets):
            rgb = rgb_cpu.unsqueeze(0).to(device)         # [1, 3, H, W]
            real_ir = real_ir_cpu.unsqueeze(0).to(device) # [1, 1, H, W] (1 KÊNH)

            if is_real_ir_anchor:
                eval_ir_1ch = real_ir
            else:
                assert translator is not None
                eval_ir_1ch = translator.translate_1ch(rgb)  # [1, 1, H, W] (ĐÚNG 1 KÊNH DO GENERATOR SINH RA)
                assert eval_ir_1ch.shape[1] == 1

            # --- Thu thập đặc trưng cho CÁCH 1: Image Generation Metrics ---
            if not is_real_ir_anchor and inception_extractor is not None:
                real_inc_list.append(inception_extractor(real_ir).cpu().numpy())
                fake_inc_list.append(inception_extractor(eval_ir_1ch).cpu().numpy())

                rgb_luma = 0.2989 * rgb[:, 0:1] + 0.5870 * rgb[:, 1:2] + 0.1140 * rgb[:, 2:3]
                g_rgb = _sobel_magnitude(rgb_luma).flatten(1)
                g_fake = _sobel_magnitude(eval_ir_1ch).flatten(1)
                edge_cos_scores.append(float(F.cosine_similarity(g_rgb, g_fake, dim=1).item()))

                real_hf_energies.append(_laplacian_energy(real_ir))
                fake_hf_energies.append(_laplacian_energy(eval_ir_1ch))

                mse = float(F.mse_loss(eval_ir_1ch, real_ir).item())
                psnr_scores.append(10.0 * math.log10(1.0 / max(mse, 1e-10)))
                ssim_scores.append(_ssim_1ch(eval_ir_1ch, real_ir))

                for _, c_r in _signed_box_ring_contrasts(real_ir[0, 0], tgt["boxes"], tgt["labels"]):
                    real_contrasts_all.append(c_r)
                for lbl_f, c_f in _signed_box_ring_contrasts(eval_ir_1ch[0, 0], tgt["boxes"], tgt["labels"]):
                    fake_contrasts_all.append(c_f)
                    if lbl_f == FLIR_CLASS_TO_IDX["person"]:
                        fake_person_contrasts.append(c_f)

            # Lặp 1 kênh -> 3 kênh y hệt training/gan_bridge.py#L46 khi đưa vào FCOS Detector
            fcos_batch_3ch.append(eval_ir_1ch.squeeze(0).repeat(3, 1, 1))

        # --- Chạy CÁCH 2: Đóng băng Best FCOS Detector ---
        if fcos_judge is not None:
            preds = fcos_judge.predict(fcos_batch_3ch)
            fcos_predictions.extend({k: v.detach().cpu() for k, v in p.items()} for p in preds)
            fcos_targets.extend(
                {k: v.detach().cpu() for k, v in t.items() if isinstance(v, Tensor)}
                for t in targets
            )

    report: Dict[str, Any] = {
        "method": method_name,
        "num_eval_images": len(val_loader.dataset),
        "elapsed_sec": round(time.time() - t0, 2),
        "generation_metrics": None,
        "fcos_detection_metrics": None,
        "pseudo_label_metrics": {},
    }

    if not is_real_ir_anchor and real_inc_list:
        real_feats = np.concatenate(real_inc_list, axis=0)
        fake_feats = np.concatenate(fake_inc_list, axis=0)
        fid_kid = compute_fid_and_kid(
            real_feats, fake_feats, kid_subsets=kid_subsets, kid_subset_size=kid_subset_size
        )
        mean_real_hf = float(np.mean(real_hf_energies))
        mean_fake_hf = float(np.mean(fake_hf_energies))
        report["generation_metrics"] = {
            **fid_kid,
            "Edge_Structure_CosSim": float(np.mean(edge_cos_scores)),
            "Thermal_Contrast_W1": _wasserstein_1d(real_contrasts_all, fake_contrasts_all),
            "Person_Positive_Contrast_Ratio": (
                float(np.mean([c > 0 for c in fake_person_contrasts])) if fake_person_contrasts else float("nan")
            ),
            "HighFreq_Energy_Ratio_Fake_over_Real": mean_fake_hf / max(mean_real_hf, 1e-12),
            "Ref_Aligned_SSIM": float(np.mean(ssim_scores)),
            "Ref_Aligned_PSNR_dB": float(np.mean(psnr_scores)),
        }

    if fcos_judge is not None and fcos_predictions:
        report["fcos_detection_metrics"] = compute_coco_detection_metrics(fcos_predictions, fcos_targets)
        for conf in pseudo_confs:
            report["pseudo_label_metrics"][f"conf_{conf:.2f}"] = compute_pseudo_label_quality(
                fcos_predictions, fcos_targets, conf_thresh=conf, iou_thresh=0.5
            )
    return report


# --- 4.4 KIỂM TRA TRƯỚC KHI CHẠY DÀI TRÊN KAGGLE (PREFLIGHT FAIL-FAST CHECK) ---
def verify_all_checkpoints_before_training(cfg: Dict[str, Any]) -> None:
    """
    Xác minh ngay lập tức rằng:
      1) GPU đã bật (nếu cấu hình device='cuda' trên Kaggle).
      2) best_fcos_checkpoint tồn tại và nạp thành công 100% trước khi tốn 1.5h train GAN!
      3) cyclegan_weights (hoặc load_existing_cyclegan_eval_json) tồn tại và hợp lệ!
    """
    if not cfg.get("strict_preflight_check", True):
        return

    errors = []
    if cfg.get("device") == "cuda" and Path("/kaggle").is_dir() and not torch.cuda.is_available():
        errors.append("Bạn đang chạy trên Kaggle nhưng CHƯA BẬT GPU (Settings -> Accelerator -> GPU T4 x2 / P100)!")

    fcos_ckpt = cfg.get("best_fcos_checkpoint")
    if not fcos_ckpt or not Path(fcos_ckpt).is_file():
        errors.append(
            f"Không tìm thấy best_fcos_checkpoint tại '{fcos_ckpt}'. "
            "Hãy điền đúng đường dẫn checkpoint FCOS tốt nhất vào Cell 1!"
        )
    else:
        test_judge = load_frozen_fcos_detector(
            fcos_ckpt, torch.device("cpu"), score_thresh=cfg["fcos_score_thresh"], fallback_img_size=cfg["fcos_fallback_img_size"]
        )
        print(f"[PREFLIGHT OK] Đã kiểm tra nạp thành công Best FCOS Detector ({test_judge.fmt_name}): {fcos_ckpt}")
        del test_judge

    has_cyc_json = bool(cfg.get("load_existing_cyclegan_eval_json") and Path(cfg["load_existing_cyclegan_eval_json"]).is_file())
    has_cyc_pth = bool(cfg.get("cyclegan_weights") and Path(cfg["cyclegan_weights"]).is_file())
    if not (has_cyc_json or has_cyc_pth):
        errors.append(
            f"Không tìm thấy cyclegan_weights='{cfg.get('cyclegan_weights')}' "
            f"(cũng chưa có load_existing_cyclegan_eval_json)."
        )
    elif has_cyc_pth:
        _ = load_gan_translator(cfg["cyclegan_weights"], torch.device("cpu"))
        print(f"[PREFLIGHT OK] Đã kiểm tra nạp thành công CycleGAN baseline: {cfg['cyclegan_weights']}")

    resume_p = cfg.get("resume_train_state")
    if resume_p:
        rp = Path(resume_p)
        has_valid_resume = rp.is_file() or (
            rp.is_dir() and (any(rp.rglob("data.pkl")) or any(c.is_file() for c in rp.rglob("*.pth")))
        )
        if not has_valid_resume:
            errors.append(f"Không tìm thấy resume_train_state hợp lệ tại '{resume_p}'!")
        else:
            kind = "thư mục Kaggle auto-unpacked (tự ghép khi nạp)" if rp.is_dir() else "file .pth"
            print(f"[PREFLIGHT OK] Đã xác nhận resume_train_state ({kind}): {resume_p}")

    if errors:
        scan_kaggle_inputs_hint()
        joined = "\n  - ".join(errors)
        raise FileNotFoundError(
            f"[PREFLIGHT DỪNG SỚM ĐỂ BẢO VỆ QUOTA GPU CỦA BẠN]:\n  - {joined}\n"
            "(Nếu muốn bỏ qua kiểm tra này, đặt CONFIG['strict_preflight_check'] = False ở Cell 1)"
        )


if DATA_ROOT.is_dir():
    verify_all_checkpoints_before_training(CONFIG)

In [ ]:
# ==============================================================================
# CELL 5: HUẤN LUYỆN UNPAIRED F-LSeSim (RGB 3 KÊNH -> IR 1 KÊNH)
# ==============================================================================
from torch.utils.data import Subset


def build_linear_decay_scheduler(optimizer: torch.optim.Optimizer, n_epochs: int, n_epochs_decay: int):
    def lr_lambda(epoch_idx: int) -> float:
        return 1.0 - max(0, epoch_idx - n_epochs) / float(max(1, n_epochs_decay + 1))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def _load_resume_checkpoint_compat(path: str | Path, map_location: Any) -> Any:
    """Nạp resume checkpoint kể cả khi Kaggle tự giải nén file .pth thành thư mục chứa data.pkl."""
    import tempfile, zipfile
    p = Path(path)
    if p.is_file():
        return torch.load(p, map_location=map_location, weights_only=False)
    if p.is_dir():
        nested = [c for c in sorted(p.rglob("*.pth")) if c.is_file()] + [c for c in sorted(p.rglob("*.pt")) if c.is_file()]
        if nested:
            return torch.load(nested[0], map_location=map_location, weights_only=False)
        pkls = sorted(p.rglob("data.pkl"))
        if pkls:
            root = pkls[0].parent
            fd, tmp_name = tempfile.mkstemp(suffix=".pth")
            os.close(fd)
            tmp_path = Path(tmp_name)
            try:
                print(f"[AUTO-REPACK] Phát hiện resume '{p.name}' là thư mục do Kaggle giải nén -> đang ghép tạm để nạp...")
                with zipfile.ZipFile(tmp_path, "w", compression=zipfile.ZIP_STORED) as zf:
                    for f in sorted(root.rglob("*")):
                        if f.is_file():
                            zf.write(f, "archive/" + f.relative_to(root).as_posix())
                return torch.load(tmp_path, map_location=map_location, weights_only=False)
            finally:
                if tmp_path.exists():
                    tmp_path.unlink()
    raise FileNotFoundError(f"Không tìm thấy resume checkpoint hợp lệ tại: {p}")


@torch.no_grad()
def _translate_rgb_to_ir1ch_eval(gen_model: nn.Module, rgb_01: Tensor) -> Tensor:
    """Chạy suy luận RGB [1,3,H,W] -> IR [1,1,H,W] trong [0,1] mà KHÔNG làm thay đổi requires_grad của gen_model."""
    height, width = rgb_01.shape[-2:]
    pad_h = max(8, (height + 3) // 4 * 4) - height
    pad_w = max(8, (width + 3) // 4 * 4) - width
    gan_input = F.pad(rgb_01 * 2.0 - 1.0, (0, pad_w, 0, pad_h), mode="replicate")
    raw_out = gen_model(gan_input)
    return ((raw_out[:, :, :height, :width] + 1.0) * 0.5).clamp(0.0, 1.0)


@torch.no_grad()
def _extract_real_ir_inception_feats(
    loader: DataLoader, inc_ext: InceptionPool3Extractor, device: torch.device
) -> np.ndarray:
    feats_list = []
    for batch in loader:
        ir_299 = [
            F.interpolate(ir_cpu.unsqueeze(0).to(device), size=(299, 299), mode="bilinear", align_corners=False)
            for ir_cpu in batch["ir_1ch"]
        ]
        feats_list.append(inc_ext(torch.cat(ir_299, dim=0)).cpu().numpy())
    return np.concatenate(feats_list, axis=0)


@torch.no_grad()
def _compute_probe_fid(
    gen_model: nn.Module,
    loader: DataLoader,
    inc_ext: InceptionPool3Extractor,
    cached_real_feats: np.ndarray,
    device: torch.device,
) -> float:
    """Đo chính xác FID trên ảnh IR 1 kênh (tái sử dụng feature Real-IR đã cache -> chỉ ~10-15s/lần trên GPU)."""
    was_training = gen_model.training
    gen_model.eval()
    try:
        fake_feats_list = []
        for batch in loader:
            fake_299 = []
            for rgb_cpu in batch["rgb"]:
                rgb = rgb_cpu.unsqueeze(0).to(device)
                fake_1ch = _translate_rgb_to_ir1ch_eval(gen_model, rgb)
                fake_299.append(F.interpolate(fake_1ch, size=(299, 299), mode="bilinear", align_corners=False))
            fake_feats_list.append(inc_ext(torch.cat(fake_299, dim=0)).cpu().numpy())
    finally:
        if was_training:
            gen_model.train()
    fake_feats = np.concatenate(fake_feats_list, axis=0)
    return float(compute_fid_and_kid(cached_real_feats, fake_feats, kid_subsets=0)["FID"])


def train_flsesim_rgb2ir(cfg: Dict[str, Any]) -> Path:
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục FLIR tại: {DATA_ROOT}")

    use_lsesim = cfg["flsesim_mode"].lower() == "lsesim"
    train_loader = DataLoader(
        train_unpaired_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=True,
        worker_init_fn=dataloader_worker_init_fn,
    )

    # 1. Khởi tạo mạng: Generator (3ch -> 1ch), Discriminator (1ch), VGG16 + SpatialCorrelativeLoss
    netG = CycleGANGenerator(ngf=64).to(DEVICE)
    netD = PatchGANDiscriminator1Ch(input_nc=1, ndf=64, n_layers=3).to(DEVICE)
    init_xavier_weights(netG, gain=0.02)
    init_xavier_weights(netD, gain=0.02)

    vgg_ext = VGG16FeatureExtractor(weights_mode="pretrained").to(DEVICE)
    img_norm = ImageNetBroadcastNorm().to(DEVICE)
    criterion_spatial = SpatialCorrelativeLoss(
        patch_nums=cfg["patch_nums"],
        patch_size=cfg["patch_size"],
        use_norm=cfg["use_norm"],
        learned_attn=use_lsesim,
        temperature=cfg["temperature"],
    ).to(DEVICE)
    criterion_gan_mse = nn.MSELoss()
    fake_ir_pool = ImagePool(pool_size=cfg["pool_size"])

    first_batch = next(iter(train_loader))
    if use_lsesim:
        with torch.no_grad():
            dummy_a = img_norm((first_batch["A"][:1].to(DEVICE) + 1.0) * 0.5)
            dummy_feats = vgg_ext(dummy_a, layers=cfg["attn_layers"])
        for idx, f in enumerate(dummy_feats):
            criterion_spatial._get_or_build_proj(f, idx)

    opt_G = torch.optim.Adam(netG.parameters(), lr=cfg["lr"], betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(netD.parameters(), lr=cfg["lr"], betas=(0.5, 0.999))
    opt_F = (
        torch.optim.Adam(criterion_spatial.projs.parameters(), lr=cfg["lr"], betas=(0.5, 0.999))
        if use_lsesim else None
    )

    total_epochs = int(cfg["n_epochs"]) + int(cfg["n_epochs_decay"])
    sched_G = build_linear_decay_scheduler(opt_G, cfg["n_epochs"], cfg["n_epochs_decay"])
    sched_D = build_linear_decay_scheduler(opt_D, cfg["n_epochs"], cfg["n_epochs_decay"])
    sched_F = build_linear_decay_scheduler(opt_F, cfg["n_epochs"], cfg["n_epochs_decay"]) if opt_F is not None else None

    start_epoch = 1
    best_probe_fid = float("inf")
    history: List[Dict[str, float]] = []

    if cfg.get("resume_train_state") and Path(cfg["resume_train_state"]).exists():
        ckpt = _load_resume_checkpoint_compat(cfg["resume_train_state"], map_location=DEVICE)
        netG.load_state_dict(ckpt["netG"])
        netD.load_state_dict(ckpt["netD"])
        opt_G.load_state_dict(ckpt["opt_G"])
        opt_D.load_state_dict(ckpt["opt_D"])
        sched_G.load_state_dict(ckpt["sched_G"])
        sched_D.load_state_dict(ckpt["sched_D"])
        if use_lsesim and ckpt.get("netF") is not None and opt_F is not None and sched_F is not None:
            criterion_spatial.projs.load_state_dict(ckpt["netF"])
            opt_F.load_state_dict(ckpt["opt_F"])
            sched_F.load_state_dict(ckpt["sched_F"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_probe_fid = float(ckpt.get("best_probe_fid", float("inf")))
        history = list(ckpt.get("history", []))
        best_fid_txt = f"{best_probe_fid:.2f}" if math.isfinite(best_probe_fid) else "chưa đo (sẽ đo mốc epoch resume ngay bây giờ)"
        print(
            f"[RESUME] Tiếp tục huấn luyện F-LSeSim từ epoch {start_epoch}/{total_epochs} "
            f"(còn {max(0, total_epochs - start_epoch + 1)} epochs | best FID={best_fid_txt})"
        )

    latest_g_path = OUTPUT_DIR / "latest_net_G_A.pth"
    best_g_path = OUTPUT_DIR / "best_net_G_A.pth"
    resume_path = OUTPUT_DIR / "flsesim_resume_latest.pth"

    # Khởi tạo bộ đo FID (cache sẵn đặc trưng Real-IR 1 lần duy nhất để chấm mỗi 10 epoch siêu nhanh!)
    probe_inc: Optional[InceptionPool3Extractor] = None
    probe_loader: Optional[DataLoader] = None
    cached_real_ir_feats: Optional[np.ndarray] = None
    if cfg.get("probe_every_epochs", 0) > 0:
        probe_inc = InceptionPool3Extractor().to(DEVICE)
        max_probe = cfg.get("probe_fid_max_images", None)
        if max_probe is not None and int(max_probe) < len(val_eval_ds):
            n_sub = int(max_probe)
            rng_idx = np.random.default_rng(cfg["seed"]).choice(len(val_eval_ds), size=n_sub, replace=False).tolist()
            probe_ds: Dataset = Subset(val_eval_ds, rng_idx)
        else:
            n_sub = len(val_eval_ds)
            probe_ds = val_eval_ds
        probe_loader = DataLoader(
            probe_ds,
            batch_size=cfg["eval_batch_size"],
            shuffle=False,
            num_workers=cfg["num_workers"],
            collate_fn=val_collate_fn,
        )
        t_cache = time.time()
        cached_real_ir_feats = _extract_real_ir_inception_feats(probe_loader, probe_inc, DEVICE)
        print(
            f"[PROBE FID] Đã cache đặc trưng InceptionV3 Real-IR ({n_sub} ảnh val, {time.time() - t_cache:.1f}s) — "
            f"sẽ đo FID mỗi {cfg['probe_every_epochs']} epochs để lưu best_net_G_A.pth (FID thấp nhất!)."
        )
        # Nếu vừa resume từ checkpoint cũ (chưa có best_probe_fid hoặc thư mục output mới), đo mốc FID hiện tại làm chuẩn so sánh!
        if start_epoch > 1 and (not math.isfinite(best_probe_fid) or not best_g_path.is_file()):
            init_fid = _compute_probe_fid(netG, probe_loader, probe_inc, cached_real_ir_feats, DEVICE)
            if init_fid < best_probe_fid or not best_g_path.is_file():
                best_probe_fid = min(best_probe_fid, init_fid)
                atomic_torch_save(netG.state_dict(), best_g_path)
            print(
                f"  ★ [MỐC FID BAN ĐẦU TỪ RESUME @ Epoch {start_epoch - 1}] FID = {init_fid:.4f} "
                f"(best FID hiện tại = {best_probe_fid:.4f}) -> đã lưu làm mốc {best_g_path.name}",
                flush=True,
            )

    print(f"[START TRAIN] F-LSeSim ({cfg['flsesim_mode'].upper()}) | Epochs: {start_epoch}..{total_epochs} | RGB(3ch) -> IR(1ch)")
    for epoch in range(start_epoch, total_epochs + 1):
        t_ep = time.time()
        netG.train()
        netD.train()
        sum_g_gan, sum_g_spa, sum_d = 0.0, 0.0, 0.0

        for batch in train_loader:
            real_A = batch["A"].to(DEVICE, non_blocking=True)  # [B, 3, H, W] in [-1, 1]
            real_B = batch["B"].to(DEVICE, non_blocking=True)  # [B, 1, H, W] in [-1, 1] (KHÔNG GHÉP CẶP VỚI A!)

            fake_B = netG(real_A)  # [B, 1, H, W] in [-1, 1] (ĐÚNG 1 KÊNH IR!)

            if use_lsesim and opt_F is not None:
                criterion_spatial.projs.requires_grad_(True)
                opt_F.zero_grad(set_to_none=True)
                norm_a = img_norm((real_A + 1.0) * 0.5)
                norm_fb_det = img_norm((fake_B.detach() + 1.0) * 0.5)
                norm_rb = img_norm((real_B + 1.0) * 0.5)
                if "A_aug" in batch and "B_aug" in batch:
                    norm_a = torch.cat([norm_a, norm_a], dim=0)
                    norm_fb_det = torch.cat([norm_fb_det, img_norm((batch["A_aug"].to(DEVICE) + 1.0) * 0.5)], dim=0)
                    norm_rb = torch.cat([norm_rb, img_norm((batch["B_aug"].to(DEVICE) + 1.0) * 0.5)], dim=0)
                f_src = vgg_ext(norm_a, layers=cfg["attn_layers"])
                f_tgt = vgg_ext(norm_fb_det, layers=cfg["attn_layers"])
                f_oth = vgg_ext(torch.flip(norm_rb, dims=[2, 3]), layers=cfg["attn_layers"])
                loss_f = criterion_spatial(f_src, f_tgt, f_oth)
                loss_f.backward()
                opt_F.step()
                criterion_spatial.projs.requires_grad_(False)

            # (1) Cập nhật Discriminator 1 kênh trên miền IR
            netD.requires_grad_(True)
            opt_D.zero_grad(set_to_none=True)
            pred_real = netD(real_B)
            pred_fake = netD(fake_ir_pool.query(fake_B))
            loss_D = 0.5 * (
                criterion_gan_mse(pred_real, torch.ones_like(pred_real))
                + criterion_gan_mse(pred_fake, torch.zeros_like(pred_fake))
            )
            loss_D.backward()
            opt_D.step()

            # (2) Cập nhật Generator 3ch -> 1ch (LSGAN + Spatially-Correlative Loss)
            netD.requires_grad_(False)
            opt_G.zero_grad(set_to_none=True)
            pred_fake_g = netD(fake_B)
            loss_G_gan = criterion_gan_mse(pred_fake_g, torch.ones_like(pred_fake_g))

            feats_real_A = vgg_ext(img_norm((real_A + 1.0) * 0.5), layers=cfg["attn_layers"])
            feats_fake_B = vgg_ext(img_norm((fake_B + 1.0) * 0.5), layers=cfg["attn_layers"])
            loss_G_spa = criterion_spatial(feats_real_A, feats_fake_B, None) * float(cfg["lambda_spatial"])

            loss_G = loss_G_gan + loss_G_spa
            loss_G.backward()
            opt_G.step()

            sum_g_gan += float(loss_G_gan.item())
            sum_g_spa += float(loss_G_spa.item())
            sum_d += float(loss_D.item())

        sched_G.step()
        sched_D.step()
        if sched_F is not None:
            sched_F.step()

        n_steps = max(1, len(train_loader))
        rec = {
            "epoch": epoch,
            "lr": float(opt_G.param_groups[0]["lr"]),
            "loss_G_GAN": sum_g_gan / n_steps,
            "loss_G_Spatial": sum_g_spa / n_steps,
            "loss_D": sum_d / n_steps,
            "sec": round(time.time() - t_ep, 1),
        }

        atomic_torch_save(netG.state_dict(), latest_g_path)
        if epoch % max(1, cfg["save_every_epochs"]) == 0 or epoch == total_epochs:
            atomic_torch_save(netG.state_dict(), OUTPUT_DIR / f"epoch_{epoch:03d}_net_G_A.pth")

        if (
            probe_inc is not None
            and probe_loader is not None
            and cached_real_ir_feats is not None
            and (epoch % cfg["probe_every_epochs"] == 0 or epoch == total_epochs)
        ):
            t_fid = time.time()
            fid_cur = _compute_probe_fid(netG, probe_loader, probe_inc, cached_real_ir_feats, DEVICE)
            rec["probe_FID"] = fid_cur
            if fid_cur < best_probe_fid:
                best_probe_fid = fid_cur
                atomic_torch_save(netG.state_dict(), best_g_path)
                print(
                    f"  ★ [NEW BEST GENERATOR @ Epoch {epoch}] Val FID = {best_probe_fid:.4f} (thấp nhất! {time.time() - t_fid:.1f}s) -> {best_g_path}",
                    flush=True,
                )
            else:
                print(
                    f"  • [PROBE FID @ Epoch {epoch}] Val FID = {fid_cur:.4f} (best hiện tại = {best_probe_fid:.4f} | {time.time() - t_fid:.1f}s)",
                    flush=True,
                )

        history.append(rec)
        atomic_torch_save(
            {
                "epoch": epoch,
                "netG": netG.state_dict(),
                "netD": netD.state_dict(),
                "netF": criterion_spatial.projs.state_dict() if use_lsesim else None,
                "opt_G": opt_G.state_dict(),
                "opt_D": opt_D.state_dict(),
                "opt_F": opt_F.state_dict() if opt_F is not None else None,
                "sched_G": sched_G.state_dict(),
                "sched_D": sched_D.state_dict(),
                "sched_F": sched_F.state_dict() if sched_F is not None else None,
                "best_probe_fid": best_probe_fid,
                "history": history,
            },
            resume_path,
        )
        probe_str = f" | Probe FID={rec['probe_FID']:.2f} (best={best_probe_fid:.2f})" if "probe_FID" in rec else ""
        print(
            f"Epoch [{epoch:03d}/{total_epochs:03d}] lr={rec['lr']:.2e} | "
            f"G_GAN={rec['loss_G_GAN']:.4f} | G_Spatial={rec['loss_G_Spatial']:.4f} | D={rec['loss_D']:.4f}"
            f"{probe_str} ({rec['sec']}s)",
            flush=True,
        )

    (OUTPUT_DIR / "flsesim_train_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

    # Giải phóng hoàn toàn bộ nhớ huấn luyện trên GPU trước khi chuyển sang bước Full Evaluation
    del netG, netD, vgg_ext, criterion_spatial, probe_inc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return best_g_path if best_g_path.is_file() else latest_g_path


TRAINED_FLSESIM_WEIGHTS: Optional[Path] = None
if CONFIG["run_training"] and DATA_ROOT.is_dir():
    TRAINED_FLSESIM_WEIGHTS = train_flsesim_rgb2ir(CONFIG)
    print(f"\n[HOÀN TẤT TRAIN F-LSeSim] Trọng số tốt nhất sẵn sàng tại: {TRAINED_FLSESIM_WEIGHTS}")
elif CONFIG.get("eval_flsesim_weights"):
    TRAINED_FLSESIM_WEIGHTS = Path(CONFIG["eval_flsesim_weights"])
    print(f"[SKIP TRAIN] Sử dụng trọng số F-LSeSim có sẵn tại: {TRAINED_FLSESIM_WEIGHTS}")

In [ ]:
# ==============================================================================
# CELL 6: ĐÁNH GIÁ KÉP SO SÁNH F-LSeSim vs CYCLEGAN & LƯU BASELINE JSON CHO NOTEBOOK 2
# ==============================================================================
def _sha256_short(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def run_full_comparison_and_export(cfg: Dict[str, Any]) -> Dict[str, Any]:
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu FLIR tại: {DATA_ROOT}")

    val_loader = DataLoader(
        val_eval_ds,
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        collate_fn=val_collate_fn,
    )
    inc_extractor = InceptionPool3Extractor().to(DEVICE)

    fcos_judge = None
    fcos_meta = None
    if cfg.get("best_fcos_checkpoint") and Path(cfg["best_fcos_checkpoint"]).is_file():
        fcos_path = Path(cfg["best_fcos_checkpoint"])
        fcos_judge = load_frozen_fcos_detector(
            fcos_path,
            DEVICE,
            score_thresh=cfg["fcos_score_thresh"],
            fallback_img_size=cfg["fcos_fallback_img_size"],
        )
        fcos_meta = {
            "checkpoint_name": fcos_path.name,
            "checkpoint_sha256_16": _sha256_short(fcos_path),
            "detector_format": fcos_judge.fmt_name,
            "score_thresh": cfg["fcos_score_thresh"],
        }
        print(f"[JUDGE] Đã đóng băng Best FCOS Detector: {fcos_path.name} ({fcos_judge.fmt_name})")
    else:
        print("[CẢNH BÁO] Không tìm thấy file 'best_fcos_checkpoint' — chỉ chạy Cách 1 (Generation Metrics).")

    results: Dict[str, Any] = {"protocol": {"split": "align_validation.txt", "num_images": len(val_eval_ds), "fcos_judge": fcos_meta}}

    # --- A. Nạp hoặc đo CycleGAN Baseline & Mốc trần Real-IR ---
    loaded_cyc = cfg.get("load_existing_cyclegan_eval_json")
    if loaded_cyc and Path(loaded_cyc).is_file():
        print(f"\n[1/3] Nạp kết quả đánh giá CycleGAN (& Real-IR nếu có) đã lưu sẵn từ: {loaded_cyc}")
        cyc_payload = json.loads(Path(loaded_cyc).read_text(encoding="utf-8"))
        results["cyclegan"] = cyc_payload["cyclegan"]
        if cyc_payload.get("real_ir_upper_bound") is not None:
            results["real_ir_upper_bound"] = cyc_payload["real_ir_upper_bound"]

    if "real_ir_upper_bound" not in results and cfg.get("eval_real_ir_upper_bound") and fcos_judge is not None:
        print("\n[1/3] Đang chấm điểm Frozen FCOS trên ảnh Real-IR thật (mốc trần tham chiếu)...")
        results["real_ir_upper_bound"] = evaluate_generator_comprehensive(
            method_name="Real_IR (Validation Anchor)",
            translator=None,
            val_loader=val_loader,
            device=DEVICE,
            inception_extractor=None,
            fcos_judge=fcos_judge,
            pseudo_confs=cfg["pseudo_label_confs"],
            is_real_ir_anchor=True,
        )

    if "cyclegan" not in results:
        if cfg.get("cyclegan_weights") and Path(cfg["cyclegan_weights"]).is_file():
            cyc_path = Path(cfg["cyclegan_weights"])
            print(f"\n[2/3] Đang đánh giá toàn diện CycleGAN hiện tại ({cyc_path.name})...")
            cyc_translator = load_gan_translator(cyc_path, DEVICE)
            cyc_report = evaluate_generator_comprehensive(
                method_name="CycleGAN (Current Baseline)",
                translator=cyc_translator,
                val_loader=val_loader,
                device=DEVICE,
                inception_extractor=inc_extractor,
                fcos_judge=fcos_judge,
                pseudo_confs=cfg["pseudo_label_confs"],
                kid_subsets=cfg["kid_subsets"],
                kid_subset_size=cfg["kid_subset_size"],
            )
            results["cyclegan"] = cyc_report
            del cyc_translator

            # LƯU FILE JSON ĐỘC LẬP CHO CYCLEGAN ĐỂ NOTEBOOK 2 DÙNG LẠI NGAY!
            export_path = Path(cfg["save_cyclegan_eval_json"])
            export_path.parent.mkdir(parents=True, exist_ok=True)
            export_payload = {
                "saved_for_reuse_in_notebook_2": True,
                "cyclegan_weights_name": cyc_path.name,
                "cyclegan_weights_sha256_16": _sha256_short(cyc_path),
                "protocol": results["protocol"],
                "real_ir_upper_bound": results.get("real_ir_upper_bound"),
                "cyclegan": cyc_report,
            }
            export_path.write_text(json.dumps(export_payload, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  ✓ [ĐÃ LƯU BASELINE CYCLEGAN CHO NOTEBOOK 2] -> {export_path}")
        else:
            print("[CẢNH BÁO] Chưa cung cấp 'cyclegan_weights' hợp lệ để đo đối chứng CycleGAN.")

    # --- C. Đánh giá F-LSeSim (nếu có cả best_net_G_A.pth và latest_net_G_A.pth, chấm bản tốt nhất trước) ---
    fl_path = TRAINED_FLSESIM_WEIGHTS
    if fl_path is not None and Path(fl_path).is_file():
        print(f"\n[3/3] Đang đánh giá toàn diện F-LSeSim ({Path(fl_path).name})...")
        fl_translator = load_gan_translator(fl_path, DEVICE)
        results["flsesim"] = evaluate_generator_comprehensive(
            method_name=f"F-LSeSim ({Path(fl_path).name})",
            translator=fl_translator,
            val_loader=val_loader,
            device=DEVICE,
            inception_extractor=inc_extractor,
            fcos_judge=fcos_judge,
            pseudo_confs=cfg["pseudo_label_confs"],
            kid_subsets=cfg["kid_subsets"],
            kid_subset_size=cfg["kid_subset_size"],
        )
        del fl_translator

    summary_path = OUTPUT_DIR / "comparison_flsesim_vs_cyclegan.json"
    summary_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n[HOÀN TẤT ĐÁNH GIÁ] Báo cáo tổng hợp đã lưu tại -> {summary_path}")
    return results


def print_side_by_side_report(results: Dict[str, Any]) -> None:
    cyc = results.get("cyclegan", {})
    fls = results.get("flsesim", {})
    real = results.get("real_ir_upper_bound", {})

    print("\n" + "=" * 92)
    print("BẢNG 1 — CÁCH EVAL 1: CÁC METRICS SINH ẢNH HỒNG NGOẠI 1 KÊNH (IMAGE GENERATION METRICS)")
    print("=" * 92)
    gen_specs = [
        ("FID (↓ càng thấp càng thật)", "FID", False),
        ("KID × 10^3 (↓ không chệch cỡ mẫu nhỏ)", "KID_x1000_mean", False),
        ("Edge Structure CosSim (↑ giữ biên vật thể)", "Edge_Structure_CosSim", True),
        ("Thermal Contrast W1 (↓ khớp tương phản nhiệt)", "Thermal_Contrast_W1", False),
        ("Person Positive Contrast (↑ người sáng hơn nền)", "Person_Positive_Contrast_Ratio", True),
        ("High-Freq Ratio Fake/Real (≈1.0 ít giấu tin)", "HighFreq_Energy_Ratio_Fake_over_Real", None),
        ("Ref Val SSIM (↑ tham chiếu)", "Ref_Aligned_SSIM", True),
        ("Ref Val PSNR dB (↑ tham chiếu)", "Ref_Aligned_PSNR_dB", True),
    ]
    print(f"{'Metric':46s} | {'CycleGAN':>12s} | {'F-LSeSim':>12s} | {'Chênh lệch (Δ)':>15s}")
    print("-" * 92)
    g_cyc = cyc.get("generation_metrics") or {}
    g_fls = fls.get("generation_metrics") or {}
    for label, key, higher_better in gen_specs:
        vc = g_cyc.get(key, float("nan"))
        vf = g_fls.get(key, float("nan"))
        delta = vf - vc if (np.isfinite(vc) and np.isfinite(vf)) else float("nan")
        verdict = ""
        if np.isfinite(delta) and higher_better is not None:
            good = (delta > 0) if higher_better else (delta < 0)
            verdict = " ✓ Tốt hơn" if good else ""
        print(f"{label:46s} | {vc:12.4f} | {vf:12.4f} | {delta:+8.4f}{verdict}")

    if any(x.get("fcos_detection_metrics") for x in (cyc, fls, real)):
        print("\n" + "=" * 98)
        print("BẢNG 2 — CÁCH EVAL 2: ĐÓNG BĂNG BEST FCOS DETECTOR CHẤM ĐIỂM DETECTION & CHẤT LƯỢNG NHÃN GIẢ")
        print("=" * 98)
        print(f"{'Metric (↑ càng cao càng tốt)':38s} | {'Real IR (Trần)':>14s} | {'CycleGAN':>12s} | {'F-LSeSim':>12s} | {'Δ vs CycleGAN':>14s}")
        print("-" * 98)
        det_keys = [
            ("COCO mAP @[0.50:0.95]", "AP"),
            ("COCO mAP @0.50 (AP50)", "AP50"),
            ("COCO mAP @0.75 (AP75)", "AP75"),
            ("AP50 / person", "AP50/person"),
            ("AP50 / car", "AP50/car"),
            ("AP50 / bicycle (vật mảnh dễ mất)", "AP50/bicycle"),
        ]
        d_real = real.get("fcos_detection_metrics") or {}
        d_cyc = cyc.get("fcos_detection_metrics") or {}
        d_fls = fls.get("fcos_detection_metrics") or {}
        for label, key in det_keys:
            vr = d_real.get(key, float("nan"))
            vc = d_cyc.get(key, float("nan"))
            vf = d_fls.get(key, float("nan"))
            dl = vf - vc if (np.isfinite(vc) and np.isfinite(vf)) else float("nan")
            mark = " ★" if (np.isfinite(dl) and dl > 0) else ""
            print(f"{label:38s} | {vr:14.4f} | {vc:12.4f} | {vf:12.4f} | {dl:+11.4f}{mark}")

        for conf_key in sorted((fls.get("pseudo_label_metrics") or cyc.get("pseudo_label_metrics") or {}).keys()):
            print("-" * 98)
            p_real = (real.get("pseudo_label_metrics") or {}).get(conf_key, {})
            p_cyc = (cyc.get("pseudo_label_metrics") or {}).get(conf_key, {})
            p_fls = (fls.get("pseudo_label_metrics") or {}).get(conf_key, {})
            for pl_label, pl_k in (
                (f"Nhãn giả Precision ({conf_key})", "PL_Precision"),
                (f"Nhãn giả Recall    ({conf_key})", "PL_Recall"),
                (f"Nhãn giả F1-Score  ({conf_key})", "PL_F1"),
                (f"Nhãn giả Mean IoU  ({conf_key})", "PL_MeanMatchedIoU"),
            ):
                vr = p_real.get(pl_k, float("nan"))
                vc = p_cyc.get(pl_k, float("nan"))
                vf = p_fls.get(pl_k, float("nan"))
                dl = vf - vc if (np.isfinite(vc) and np.isfinite(vf)) else float("nan")
                mark = " ★" if (np.isfinite(dl) and dl > 0) else ""
                print(f"{pl_label:38s} | {vr:14.4f} | {vc:12.4f} | {vf:12.4f} | {dl:+11.4f}{mark}")
        print("=" * 98)


EVAL_RESULTS: Optional[Dict[str, Any]] = None
if DATA_ROOT.is_dir():
    EVAL_RESULTS = run_full_comparison_and_export(CONFIG)
    print_side_by_side_report(EVAL_RESULTS)

In [ ]:
# ==============================================================================
# CELL 7: TRỰC QUAN HÓA ẢNH IR 1 KÊNH SINH RA & HỘP DỰ ĐOÁN CỦA FROZEN FCOS DETECTOR
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def _draw_boxes(ax, boxes: Tensor, labels: Tensor, scores: Optional[Tensor] = None, conf_min: float = 0.4):
    palette = {0: "#00ff66", 1: "#ffcc00", 2: "#00e5ff"}  # person=xanh lá, car=vàng, bicycle=xanh ngọc
    for idx in range(boxes.shape[0]):
        if scores is not None and float(scores[idx]) < conf_min:
            continue
        x1, y1, x2, y2 = boxes[idx].tolist()
        cls_id = int(labels[idx])
        color = palette.get(cls_id, "red")
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.4, edgecolor=color, facecolor="none"))
        cls_name = FLIR_CLASSES[cls_id] if 0 <= cls_id < len(FLIR_CLASSES) else str(cls_id)
        caption = f"{cls_name}" if scores is None else f"{cls_name}:{float(scores[idx]):.2f}"
        ax.text(x1, max(10, y1 - 3), caption, color="black", fontsize=7, bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))


if DATA_ROOT.is_dir():
    cyc_trans = load_gan_translator(CONFIG["cyclegan_weights"], DEVICE) if (CONFIG.get("cyclegan_weights") and Path(CONFIG["cyclegan_weights"]).is_file()) else None
    fls_trans = load_gan_translator(TRAINED_FLSESIM_WEIGHTS, DEVICE) if (TRAINED_FLSESIM_WEIGHTS and Path(TRAINED_FLSESIM_WEIGHTS).is_file()) else None
    viz_judge = (
        load_frozen_fcos_detector(
            CONFIG["best_fcos_checkpoint"], DEVICE, score_thresh=0.3, fallback_img_size=CONFIG["fcos_fallback_img_size"]
        )
        if (CONFIG.get("best_fcos_checkpoint") and Path(CONFIG["best_fcos_checkpoint"]).is_file())
        else None
    )

    chosen_indices = []
    for i in range(len(val_eval_ds)):
        lbls = set(val_eval_ds[i]["target"]["labels"].tolist())
        if FLIR_CLASS_TO_IDX["bicycle"] in lbls or FLIR_CLASS_TO_IDX["person"] in lbls:
            chosen_indices.append(i)
        if len(chosen_indices) >= CONFIG["num_visual_samples"]:
            break

    if chosen_indices:
        n_rows = len(chosen_indices)
        fig, axes = plt.subplots(n_rows, 4, figsize=(22, 4.4 * n_rows), squeeze=False)
        for r, ds_idx in enumerate(chosen_indices):
            item = val_eval_ds[ds_idx]
            rgb = item["rgb"].to(DEVICE)          # [3, H, W]
            real_ir = item["ir_1ch"].to(DEVICE)   # [1, H, W]
            tgt = item["target"]

            axes[r, 0].imshow(rgb.cpu().permute(1, 2, 0).numpy().clip(0, 1))
            _draw_boxes(axes[r, 0], tgt["boxes"], tgt["labels"])
            axes[r, 0].set_title(f"1. RGB + GT Boxes\n{item['stem']}", fontsize=9)

            axes[r, 1].imshow(real_ir[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
            if viz_judge is not None:
                p_real = viz_judge.predict([real_ir.repeat(3, 1, 1)])[0]
                _draw_boxes(axes[r, 1], p_real["boxes"].cpu(), p_real["labels"].cpu(), p_real["scores"].cpu(), conf_min=0.4)
            axes[r, 1].set_title("2. Real IR (1-ch) + Frozen FCOS", fontsize=9)

            if cyc_trans is not None:
                cyc_1ch = cyc_trans.translate_1ch(rgb.unsqueeze(0))[0]
                axes[r, 2].imshow(cyc_1ch[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
                if viz_judge is not None:
                    p_cyc = viz_judge.predict([cyc_1ch.repeat(3, 1, 1)])[0]
                    _draw_boxes(axes[r, 2], p_cyc["boxes"].cpu(), p_cyc["labels"].cpu(), p_cyc["scores"].cpu(), conf_min=0.4)
            axes[r, 2].set_title("3. CycleGAN IR (1-ch) + Frozen FCOS", fontsize=9)

            if fls_trans is not None:
                fls_1ch = fls_trans.translate_1ch(rgb.unsqueeze(0))[0]
                axes[r, 3].imshow(fls_1ch[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
                if viz_judge is not None:
                    p_fls = viz_judge.predict([fls_1ch.repeat(3, 1, 1)])[0]
                    _draw_boxes(axes[r, 3], p_fls["boxes"].cpu(), p_fls["labels"].cpu(), p_fls["scores"].cpu(), conf_min=0.4)
            axes[r, 3].set_title("4. F-LSeSim IR (1-ch) + Frozen FCOS", fontsize=9)

            for c in range(4):
                axes[r, c].axis("off")

        plt.tight_layout()
        viz_out = OUTPUT_DIR / "flsesim_vs_cyclegan_fcos_probe.png"
        fig.savefig(viz_out, dpi=150, bbox_inches="tight")
        print(f"Đã lưu ảnh đối chiếu trực quan -> {viz_out}")
        plt.show()